In [1]:
import gc
import re
from pathlib import Path

import numpy as np
import polars as pl
from unidecode import unidecode

In [2]:
from pathlib import Path

# Change this to your challenge dataset folder
DATA_ROOT = Path(r".\dataset")

# Input files
TRAIN_SOURCE1 = DATA_ROOT / "train/train_source1.tsv"
TRAIN_SOURCE2 = DATA_ROOT / "train/train_source2.tsv"
TRAIN_SOURCE3 = DATA_ROOT / "train/train_source3.tsv"

GROUND_TRUTH = DATA_ROOT / "train/train_ground_truth.tsv"

TEST_SOURCE1 = DATA_ROOT / "test/test_source1.tsv"
TEST_SOURCE2 = DATA_ROOT / "test/test_source2.tsv"
TEST_SOURCE3 = DATA_ROOT / "test/test_source3.tsv"

# Working directories
WORK_DIR = DATA_ROOT / "work"
NORMALIZED_DIR = WORK_DIR / "normalized"
CANDIDATE_DIR = WORK_DIR / "candidates"
OUTPUT_DIR = DATA_ROOT / "submission"

# Create working directories
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("WORK_DIR:", WORK_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_ROOT: dataset
WORK_DIR: dataset\work
OUTPUT_DIR: dataset\submission


In [3]:
files_to_check = [
    TRAIN_SOURCE1,
    TRAIN_SOURCE2,
    TRAIN_SOURCE3,
    GROUND_TRUTH,
    TEST_SOURCE1,
    TEST_SOURCE2,
    TEST_SOURCE3,
]

for path in files_to_check:
    if path.exists():
        size_gb = path.stat().st_size / (1024 ** 3)
        print(f"✓ {path.name:<25} {size_gb:>8.3f} GB")
    else:
        print(f"✗ MISSING: {path}")

✓ train_source1.tsv            0.196 GB
✓ train_source2.tsv            0.456 GB
✓ train_source3.tsv            0.469 GB
✓ train_ground_truth.tsv       0.118 GB
✓ test_source1.tsv             0.163 GB
✓ test_source2.tsv             0.474 GB
✓ test_source3.tsv             0.471 GB


In [4]:
sample = pl.read_csv(
    TRAIN_SOURCE1,
    separator="\t",
    n_rows=5,
)

print("Columns:")
print(sample.columns)

print("\nSample:")
print(sample)

Columns:
['entity_id', 'business_name', 'business_address', 'country']

Sample:
shape: (5, 4)
┌──────────────┬─────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name           ┆ business_address                ┆ country │
│ ---          ┆ ---                     ┆ ---                             ┆ ---     │
│ str          ┆ str                     ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop     ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money             ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc           ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
│ S1-133037285 ┆ Christ Chapel           ┆ 2100 Cameron Drive, Unit APART… ┆ US      │
│ S1-755362802 ┆ Prabhav Business Center ┆ 797, Lake Town Block A, Kolkat… ┆ India   │
└──────────────┴────────────────────

In [5]:
for path in [TRAIN_SOURCE1, TRAIN_SOURCE2, TRAIN_SOURCE3]:
    sample = pl.read_csv(
        path,
        separator="\t",
        n_rows=3,
    )

    print(f"\n{'=' * 60}")
    print(path.name)
    print("Columns:", sample.columns)
    print(sample)


train_source1.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']
shape: (3, 4)
┌──────────────┬─────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name       ┆ business_address                ┆ country │
│ ---          ┆ ---                 ┆ ---                             ┆ ---     │
│ str          ┆ str                 ┆ str                             ┆ str     │
╞══════════════╪═════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money         ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc       ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
└──────────────┴─────────────────────┴─────────────────────────────────┴─────────┘

train_source2.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']
shape: (3, 4)
┌──────────────┬─────────────────────────────

In [6]:
for path in [TRAIN_SOURCE1, TRAIN_SOURCE2, TRAIN_SOURCE3]:
    sample = pl.read_csv(
        path,
        separator="\t",
        n_rows=3,
    )

    print(f"\n{'=' * 60}")
    print(path.name)
    print("Columns:", sample.columns)
    print(sample)


train_source1.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']
shape: (3, 4)
┌──────────────┬─────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name       ┆ business_address                ┆ country │
│ ---          ┆ ---                 ┆ ---                             ┆ ---     │
│ str          ┆ str                 ┆ str                             ┆ str     │
╞══════════════╪═════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money         ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc       ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
└──────────────┴─────────────────────┴─────────────────────────────────┴─────────┘

train_source2.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']
shape: (3, 4)
┌──────────────┬─────────────────────────────

In [7]:
gt_sample = pl.read_csv(
    GROUND_TRUTH,
    separator="\t",
    n_rows=5,
)

print("Columns:")
print(gt_sample.columns)

print("\nSample:")
print(gt_sample)

Columns:
['source1_entity_id', 'matched_entity_ids']

Sample:
shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘


In [8]:
def count_rows(path):
    return (
        pl.scan_csv(
            path,
            separator="\t",
            infer_schema_length=1000,
        )
        .select(pl.len())
        .collect()
        .item()
    )


for path in files_to_check:
    if path.exists():
        count = count_rows(path)
        print(f"{path.name:<25} {count:>15,} rows")

train_source1.tsv               2,206,821 rows
train_source2.tsv               5,034,616 rows
train_source3.tsv               5,285,603 rows
train_ground_truth.tsv          2,206,821 rows
test_source1.tsv                1,732,544 rows
test_source2.tsv                4,887,273 rows
test_source3.tsv                5,082,316 rows


In [9]:
for path in [TRAIN_SOURCE1, TRAIN_SOURCE2, TRAIN_SOURCE3]:
    sample = pl.read_csv(
        path,
        separator="\t",
        n_rows=10000,
        infer_schema_length=10000,
    )

    print(f"\n{'=' * 60}")
    print(path.name)

    print("\nSchema:")
    print(sample.schema)

    print("\nNull counts:")
    print(sample.null_count())

    print("\nEmpty-string counts:")
    for col in ["business_name", "business_address", "country"]:
        if col in sample.columns:
            count = sample.select(
                (pl.col(col).fill_null("") == "").sum()
            ).item()
            print(f"  {col}: {count:,}")


train_source1.tsv

Schema:
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

Null counts:
shape: (1, 4)
┌───────────┬───────────────┬──────────────────┬─────────┐
│ entity_id ┆ business_name ┆ business_address ┆ country │
│ ---       ┆ ---           ┆ ---              ┆ ---     │
│ u32       ┆ u32           ┆ u32              ┆ u32     │
╞═══════════╪═══════════════╪══════════════════╪═════════╡
│ 0         ┆ 0             ┆ 0                ┆ 0       │
└───────────┴───────────────┴──────────────────┴─────────┘

Empty-string counts:
  business_name: 0
  business_address: 0
  country: 0

train_source2.tsv

Schema:
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

Null counts:
shape: (1, 4)
┌───────────┬───────────────┬──────────────────┬─────────┐
│ entity_id ┆ business_name ┆ business_address ┆ country │
│ ---       ┆ ---           ┆ ---              ┆ ---     │
│ u32       ┆ u32 

In [10]:
for path in [TRAIN_SOURCE1, TRAIN_SOURCE2, TRAIN_SOURCE3]:
    sample = pl.read_csv(
        path,
        separator="\t",
        columns=["country"],
        n_rows=100000,
    )

    print(f"\n{path.name}")
    print(
        sample
        .with_columns(
            pl.col("country")
            .fill_null("<NULL>")
            .str.strip_chars()
            .str.to_lowercase()
            .alias("country")
        )
        .group_by("country")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .head(20)
    )


train_source1.tsv
shape: (2, 2)
┌─────────┬───────┐
│ country ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ us      ┆ 59890 │
│ india   ┆ 40110 │
└─────────┴───────┘

train_source2.tsv
shape: (2, 2)
┌─────────┬───────┐
│ country ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ us      ┆ 59936 │
│ india   ┆ 40064 │
└─────────┴───────┘

train_source3.tsv
shape: (2, 2)
┌─────────┬───────┐
│ country ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ us      ┆ 59529 │
│ india   ┆ 40471 │
└─────────┴───────┘


In [11]:
def normalize_lazy(lf):
    return (
        lf
        .with_columns([
            # Business name
            pl.col("business_name")
            .fill_null("")
            .str.normalize("NFKC")
            .str.to_lowercase()
            .str.replace_all(r"[\p{P}\p{S}]", " ")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
            .alias("name_norm"),

            # Business address
            pl.col("business_address")
            .fill_null("")
            .str.normalize("NFKC")
            .str.to_lowercase()
            .str.replace_all(r"[\p{P}\p{S}]", " ")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
            .alias("address_norm"),

            # Country
            pl.col("country")
            .fill_null("")
            .str.normalize("NFKC")
            .str.to_lowercase()
            .str.strip_chars()
            .alias("country_norm"),
        ])
    )

In [12]:
test_df = pl.DataFrame({
    "business_name": [
        "Maure Williams Colombier, Inc.",
        "राम मार्केटिंग प्राइवेट लिमिटेड",
        "ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி",
        "Burger-Solution Limited",
    ],
    "business_address": [
        "123, Main Street.",
        "दिल्ली, भारत",
        "சென்னை, தமிழ்நாடு",
        "45-B Market Road",
    ],
    "country": [
        "USA",
        "India",
        "India",
        "India",
    ],
})

test_normalized = normalize_lazy(test_df)

print(test_normalized)

shape: (4, 6)
┌──────────────────┬──────────────────┬─────────┬─────────────────┬─────────────────┬──────────────┐
│ business_name    ┆ business_address ┆ country ┆ name_norm       ┆ address_norm    ┆ country_norm │
│ ---              ┆ ---              ┆ ---     ┆ ---             ┆ ---             ┆ ---          │
│ str              ┆ str              ┆ str     ┆ str             ┆ str             ┆ str          │
╞══════════════════╪══════════════════╪═════════╪═════════════════╪═════════════════╪══════════════╡
│ Maure Williams   ┆ 123, Main        ┆ USA     ┆ maure williams  ┆ 123 main street ┆ usa          │
│ Colombier, Inc.  ┆ Street.          ┆         ┆ colombier inc   ┆                 ┆              │
│ राम मार्केटिंग      ┆ दिल्ली, भारत      ┆ India   ┆ राम मार्केटिंग     ┆ दिल्ली भारत      ┆ india        │
│ प्राइवेट लिमिटे…    ┆                  ┆         ┆ प्राइवेट लिमिटे…   ┆                 ┆              │
│ ராஜ் இன்வெஸ்ட்மெண்ட்ஸ்   ┆ சென்னை, தமிழ்நாடு   ┆ India   ┆ ரா

In [13]:
for row in test_normalized.iter_rows(named=True):
    print("Original :", row["business_name"])
    print("Normalized:", row["name_norm"])
    print()

Original : Maure Williams Colombier, Inc.
Normalized: maure williams colombier inc

Original : राम मार्केटिंग प्राइवेट लिमिटेड
Normalized: राम मार्केटिंग प्राइवेट लिमिटेड

Original : ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி
Normalized: ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி

Original : Burger-Solution Limited
Normalized: burger solution limited



In [14]:
def process_source(input_path, output_path):
    print(f"\nProcessing: {input_path.name}")

    # Lazy scan of the TSV
    lf = pl.scan_csv(
        input_path,
        separator="\t",
        infer_schema_length=10000,
    )

    # Normalize fields
    lf = normalize_lazy(lf)

    # Materialize only this source
    df = lf.collect()

    print(f"Rows loaded: {len(df):,}")

    # Keep only the fields we need at this stage
    df = df.select([
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "name_norm",
        "address_norm",
        "country_norm",
    ])

    # Write compressed Parquet
    df.write_parquet(
        output_path,
        compression="zstd",
    )

    print(f"Written: {output_path}")
    print(f"Size: {output_path.stat().st_size / (1024 ** 2):.2f} MB")

    # Release memory
    del df
    gc.collect()

In [15]:
NORM_TRAIN_S1 = NORMALIZED_DIR / "train_source1.parquet"
NORM_TRAIN_S2 = NORMALIZED_DIR / "train_source2.parquet"
NORM_TRAIN_S3 = NORMALIZED_DIR / "train_source3.parquet"

NORM_TEST_S1 = NORMALIZED_DIR / "test_source1.parquet"
NORM_TEST_S2 = NORMALIZED_DIR / "test_source2.parquet"
NORM_TEST_S3 = NORMALIZED_DIR / "test_source3.parquet"

In [16]:
process_source(
    TRAIN_SOURCE1,
    NORM_TRAIN_S1,
)


Processing: train_source1.tsv
Rows loaded: 2,206,821
Written: dataset\work\normalized\train_source1.parquet
Size: 141.27 MB


In [17]:
process_source(
    TRAIN_SOURCE2,
    NORM_TRAIN_S2,
)
process_source(
    TRAIN_SOURCE3,
    NORM_TRAIN_S3,
)


Processing: train_source2.tsv
Rows loaded: 5,034,616
Written: dataset\work\normalized\train_source2.parquet
Size: 341.39 MB

Processing: train_source3.tsv
Rows loaded: 5,285,603
Written: dataset\work\normalized\train_source3.parquet
Size: 352.02 MB


In [18]:
normalized_files = [
    NORM_TRAIN_S1,
    NORM_TRAIN_S2,
    NORM_TRAIN_S3,
]

for path in normalized_files:
    print(
        f"{path.name:<30} "
        f"{path.stat().st_size / (1024**2):>10.2f} MB"
    )

train_source1.parquet              141.27 MB
train_source2.parquet              341.39 MB
train_source3.parquet              352.02 MB


In [19]:
for path in normalized_files:
    df = pl.read_parquet(path, n_rows=5)

    print(f"\n{'=' * 70}")
    print(path.name)
    print(df.schema)
    print()
    print(df)


train_source1.parquet
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String, 'name_norm': String, 'address_norm': String, 'country_norm': String})

shape: (5, 7)
┌──────────────┬──────────────┬──────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ entity_id    ┆ business_nam ┆ business_add ┆ country ┆ name_norm    ┆ address_norm ┆ country_nor │
│ ---          ┆ e            ┆ ress         ┆ ---     ┆ ---          ┆ ---          ┆ m           │
│ str          ┆ ---          ┆ ---          ┆ str     ┆ str          ┆ str          ┆ ---         │
│              ┆ str          ┆ str          ┆         ┆              ┆              ┆ str         │
╞══════════════╪══════════════╪══════════════╪═════════╪══════════════╪══════════════╪═════════════╡
│ S1-925783039 ┆ Orelee's     ┆ 1795         ┆ US      ┆ orelee s     ┆ 1795         ┆ us          │
│              ┆ Barbershop   ┆ Westchester  ┆         ┆ barbershop   ┆ westchester 

In [20]:
for path in normalized_files:
    df = pl.read_parquet(
        path,
        columns=[
            "entity_id",
            "business_name",
            "name_norm",
        ],
        n_rows=100000,
    )

    multilingual = df.filter(
        pl.col("name_norm").str.contains(
            r"[^\x00-\x7F]"
        )
    )

    print(f"\n{path.name}")
    print(f"Non-ASCII records in sample: {len(multilingual):,}")
    print(
        multilingual
        .select([
            "entity_id",
            "business_name",
            "name_norm",
        ])
        .head(10)
    )


train_source1.parquet
Non-ASCII records in sample: 0
shape: (0, 3)
┌───────────┬───────────────┬───────────┐
│ entity_id ┆ business_name ┆ name_norm │
│ ---       ┆ ---           ┆ ---       │
│ str       ┆ str           ┆ str       │
╞═══════════╪═══════════════╪═══════════╡
└───────────┴───────────────┴───────────┘

train_source2.parquet
Non-ASCII records in sample: 15,061
shape: (10, 3)
┌──────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ entity_id    ┆ business_name                   ┆ name_norm                       │
│ ---          ┆ ---                             ┆ ---                             │
│ str          ┆ str                             ┆ str                             │
╞══════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ S2-166376419 ┆ राम मार्केटिंग प्राइवेट लिमिटे…       ┆ राम मार्केटिंग प्राइवेट लिमिटे…       │
│ S2-639257739 ┆ आदित्य प्रॉपर्टीज एलएलपी           ┆ आदित्य प्रॉपर्टीज एलएलपी      

In [21]:
for path in normalized_files:
    df = pl.read_parquet(
        path,
        columns=["entity_id"],
    )

    print(f"\n{path.name}")

    print(
        df
        .with_columns(
            pl.col("entity_id")
            .str.extract(r"^(S\d+)", 1)
            .alias("source")
        )
        .group_by("source")
        .agg(pl.len().alias("count"))
        .sort("source")
    )


train_source1.parquet
shape: (1, 2)
┌────────┬─────────┐
│ source ┆ count   │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ S1     ┆ 2206821 │
└────────┴─────────┘

train_source2.parquet
shape: (1, 2)
┌────────┬─────────┐
│ source ┆ count   │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ S2     ┆ 5034616 │
└────────┴─────────┘

train_source3.parquet
shape: (1, 2)
┌────────┬─────────┐
│ source ┆ count   │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ S3     ┆ 5285603 │
└────────┴─────────┘


In [22]:
def add_transliteration(df):
    return df.with_columns([
        pl.col("name_norm")
        .map_elements(
            lambda x: unidecode(x) if x else "",
            return_dtype=pl.String,
        )
        .alias("name_translit"),

        pl.col("address_norm")
        .map_elements(
            lambda x: unidecode(x) if x else "",
            return_dtype=pl.String,
        )
        .alias("address_translit"),
    ])


In [23]:
sample = pl.read_parquet(
    NORM_TRAIN_S2,
    n_rows=20,
)

sample = add_transliteration(sample)

print(
    sample.select([
        "business_name",
        "name_norm",
        "name_translit",
    ])
)

shape: (20, 3)
┌──────────────────────────────┬──────────────────────────────┬─────────────────────────────────┐
│ business_name                ┆ name_norm                    ┆ name_translit                   │
│ ---                          ┆ ---                          ┆ ---                             │
│ str                          ┆ str                          ┆ str                             │
╞══════════════════════════════╪══════════════════════════════╪═════════════════════════════════╡
│ राम मार्केटिंग प्राइवेट लिमिटे…    ┆ राम मार्केटिंग प्राइवेट लिमिटे…    ┆ raam maarkettiNg praaivett lim… │
│ -- Holloway Peak Inc Seafood ┆ holloway peak inc seafood    ┆ holloway peak inc seafood       │
│ आदित्य प्रॉपर्टीज एलएलपी        ┆ आदित्य प्रॉपर्टीज एलएलपी        ┆ aadity proNprttiij elelpii      │
│ Summit Inc                   ┆ summit inc                   ┆ summit inc                      │
│ Delta Tetlecommunication Inc ┆ delta tetlecommunication inc ┆ delta tetlecommunicat

In [24]:
def add_transliteration_to_parquet(path):
    print(f"\nProcessing: {path.name}")

    df = pl.read_parquet(path)

    print(f"Rows: {len(df):,}")

    df = add_transliteration(df)

    df.write_parquet(
        path,
        compression="zstd",
    )

    print(f"Updated: {path}")

    del df
    gc.collect()

In [25]:

add_transliteration_to_parquet(NORM_TRAIN_S1)
add_transliteration_to_parquet(NORM_TRAIN_S2)
add_transliteration_to_parquet(NORM_TRAIN_S3)


Processing: train_source1.parquet
Rows: 2,206,821
Updated: dataset\work\normalized\train_source1.parquet

Processing: train_source2.parquet
Rows: 5,034,616
Updated: dataset\work\normalized\train_source2.parquet

Processing: train_source3.parquet
Rows: 5,285,603
Updated: dataset\work\normalized\train_source3.parquet


In [26]:
df = pl.read_parquet(
    NORM_TRAIN_S2,
    columns=[
        "entity_id",
        "business_name",
        "name_norm",
        "name_translit",
        "business_address",
        "address_norm",
        "address_translit",
    ],
    n_rows=10000,
)

print(df.schema)

print("\nSample:")
print(
    df.select([
        "business_name",
        "name_norm",
        "name_translit",
    ]).head(10)
)
multilingual = df.filter(
    pl.col("name_norm").str.contains(r"[^\x00-\x7F]")
)

print(f"Multilingual records: {len(multilingual):,}")

print(
    multilingual.select([
        "business_name",
        "name_norm",
        "name_translit",
    ]).head(20)
)

Schema({'entity_id': String, 'business_name': String, 'name_norm': String, 'name_translit': String, 'business_address': String, 'address_norm': String, 'address_translit': String})

Sample:
shape: (10, 3)
┌────────────────────────────────┬────────────────────────────────┬────────────────────────────────┐
│ business_name                  ┆ name_norm                      ┆ name_translit                  │
│ ---                            ┆ ---                            ┆ ---                            │
│ str                            ┆ str                            ┆ str                            │
╞════════════════════════════════╪════════════════════════════════╪════════════════════════════════╡
│ राम मार्केटिंग प्राइवेट लिमिटे…      ┆ राम मार्केटिंग प्राइवेट लिमिटे…      ┆ raam maarkettiNg praaivett     │
│                                ┆                                ┆ lim…                           │
│ -- Holloway Peak Inc Seafood   ┆ holloway peak inc seafood      ┆ holloway

In [27]:
gt = pl.read_csv(
    GROUND_TRUTH,
    separator="\t",
)

print(f"Ground-truth rows: {len(gt):,}")
print(gt.head())
print(gt.schema)

Ground-truth rows: 2,206,821
shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘
Schema({'source1_entity_id': String, 'matched_entity_ids': String})


## 25

In [28]:
gt_pairs = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .fill_null("")
        .str.split(",")
        .alias("matched_ids")
    )
    .explode("matched_ids")
    .rename({
        "matched_ids": "candidate_entity_id"
    })
    .filter(
        pl.col("candidate_entity_id").str.len_chars() > 0
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id",
    ])
)

C:\Users\KRISH\tmp\ipykernel_22912\498163265.py:9: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("matched_ids")


In [29]:
print(f"Total ground-truth pairs: {len(gt_pairs):,}")
print(gt_pairs.head(20))

Total ground-truth pairs: 7,638,365
shape: (20, 2)
┌───────────────────┬─────────────────────┐
│ source1_entity_id ┆ candidate_entity_id │
│ ---               ┆ ---                 │
│ str               ┆ str                 │
╞═══════════════════╪═════════════════════╡
│ S1-965667         ┆ S2-681193310        │
│ S1-965667         ┆ S2-743505751        │
│ S1-965667         ┆ S3-775321672        │
│ S1-965667         ┆ S3-11291185         │
│ S1-965667         ┆ S3-860443364        │
│ …                 ┆ …                   │
│ S1-102811957      ┆ S2-478959098        │
│ S1-102811957      ┆ S2-553508714        │
│ S1-102811957      ┆ S2-625774905        │
│ S1-102811957      ┆ S3-728090388        │
│ S1-102811957      ┆ S3-928796641        │
└───────────────────┴─────────────────────┘


In [30]:
gt_s2 = gt_pairs.filter(
    pl.col("candidate_entity_id").str.starts_with("S2-")
)

gt_s3 = gt_pairs.filter(
    pl.col("candidate_entity_id").str.starts_with("S3-")
)

print(f"S1 → S2 true pairs: {len(gt_s2):,}")
print(f"S1 → S3 true pairs: {len(gt_s3):,}")

S1 → S2 true pairs: 3,693,619
S1 → S3 true pairs: 3,944,746


In [31]:
match_counts = (
    gt_pairs
    .group_by("source1_entity_id")
    .agg(
        pl.len().alias("true_match_count")
    )
)

print(match_counts.describe())

print(
    match_counts
    .group_by("true_match_count")
    .agg(pl.len().alias("source1_count"))
    .sort("true_match_count")
    .head(20)
)

shape: (9, 3)
┌────────────┬───────────────────┬──────────────────┐
│ statistic  ┆ source1_entity_id ┆ true_match_count │
│ ---        ┆ ---               ┆ ---              │
│ str        ┆ str               ┆ f64              │
╞════════════╪═══════════════════╪══════════════════╡
│ count      ┆ 2083574           ┆ 2.083574e6       │
│ null_count ┆ 0                 ┆ 0.0              │
│ mean       ┆ null              ┆ 3.665992         │
│ std        ┆ null              ┆ 1.526294         │
│ min        ┆ S1-100000221      ┆ 1.0              │
│ 25%        ┆ null              ┆ 3.0              │
│ 50%        ┆ null              ┆ 4.0              │
│ 75%        ┆ null              ┆ 5.0              │
│ max        ┆ S1-99999926       ┆ 11.0             │
└────────────┴───────────────────┴──────────────────┘
shape: (11, 2)
┌──────────────────┬───────────────┐
│ true_match_count ┆ source1_count │
│ ---              ┆ ---           │
│ u32              ┆ u32           │
╞════════════

In [32]:
s1_ids = pl.read_parquet(
    NORM_TRAIN_S1,
    columns=["entity_id"],
)

s2_ids = pl.read_parquet(
    NORM_TRAIN_S2,
    columns=["entity_id"],
)

s3_ids = pl.read_parquet(
    NORM_TRAIN_S3,
    columns=["entity_id"],
)

print(
    "Missing S1 IDs:",
    len(
        gt
        .join(
            s1_ids,
            left_on="source1_entity_id",
            right_on="entity_id",
            how="anti",
        )
    )
)

print(
    "Missing S2 IDs:",
    len(
        gt_s2
        .join(
            s2_ids,
            left_on="candidate_entity_id",
            right_on="entity_id",
            how="anti",
        )
    )
)

print(
    "Missing S3 IDs:",
    len(
        gt_s3
        .join(
            s3_ids,
            left_on="candidate_entity_id",
            right_on="entity_id",
            how="anti",
        )
    )
)

Missing S1 IDs: 0
Missing S2 IDs: 0
Missing S3 IDs: 0


In [33]:
BLOCK_COLS = [
    "entity_id",
    "name_norm",
    "name_translit",
    "address_norm",
    "address_translit",
    "country_norm",
]

s1 = pl.read_parquet(
    NORM_TRAIN_S1,
    columns=BLOCK_COLS,
)

s2 = pl.read_parquet(
    NORM_TRAIN_S2,
    columns=BLOCK_COLS,
)

print(f"S1 rows: {len(s1):,}")
print(f"S2 rows: {len(s2):,}")

S1 rows: 2,206,821
S2 rows: 5,034,616


In [34]:
#30

s1_name_block = (
    s1
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        "name_norm",
    ])
    .filter(
        pl.col("name_norm").str.len_chars() > 0
    )
)

s2_name_block = (
    s2
    .select([
        pl.col("entity_id").alias("candidate_entity_id"),
        "name_norm",
    ])
    .filter(
        pl.col("name_norm").str.len_chars() > 0
    )
)

In [35]:
exact_name_candidates = (
    s1_name_block
    .join(
        s2_name_block,
        on="name_norm",
        how="inner",
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id",
    ])
    .unique()
)

print(
    f"Exact-name candidate pairs: "
    f"{len(exact_name_candidates):,}"
)

Exact-name candidate pairs: 10,346,281


In [36]:
exact_name_recovered = (
    gt_s2
    .join(
        exact_name_candidates,
        on=[
            "source1_entity_id",
            "candidate_entity_id",
        ],
        how="inner",
    )
)

exact_name_recall = (
    len(exact_name_recovered) / len(gt_s2)
)

print(f"True S1→S2 pairs:      {len(gt_s2):,}")
print(f"Recovered pairs:       {len(exact_name_recovered):,}")
print(f"Blocking recall:       {exact_name_recall:.4%}")

True S1→S2 pairs:      3,693,619
Recovered pairs:       792,009
Blocking recall:       21.4426%


In [37]:
exact_name_stats = (
    exact_name_candidates
    .group_by("source1_entity_id")
    .agg(
        pl.len().alias("candidate_count")
    )
)

print(
    exact_name_stats.select([
        pl.col("candidate_count").mean().alias("mean"),
        pl.col("candidate_count").median().alias("median"),
        pl.col("candidate_count").quantile(0.95).alias("p95"),
        pl.col("candidate_count").max().alias("max"),
    ])
)

shape: (1, 4)
┌──────────┬────────┬──────┬─────┐
│ mean     ┆ median ┆ p95  ┆ max │
│ ---      ┆ ---    ┆ ---  ┆ --- │
│ f64      ┆ f64    ┆ f64  ┆ u32 │
╞══════════╪════════╪══════╪═════╡
│ 8.841675 ┆ 2.0    ┆ 46.0 ┆ 526 │
└──────────┴────────┴──────┴─────┘


In [38]:
exact_name_missed = (
    gt_s2
    .join(
        exact_name_candidates,
        on=[
            "source1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
)

print(f"Missed true pairs: {len(exact_name_missed):,}")
print(exact_name_missed.head(20))

Missed true pairs: 2,901,610
shape: (20, 2)
┌───────────────────┬─────────────────────┐
│ source1_entity_id ┆ candidate_entity_id │
│ ---               ┆ ---                 │
│ str               ┆ str                 │
╞═══════════════════╪═════════════════════╡
│ S1-965667         ┆ S2-681193310        │
│ S1-965667         ┆ S2-743505751        │
│ S1-55344266       ┆ S2-249013014        │
│ S1-343815751      ┆ S2-790675320        │
│ S1-343815751      ┆ S2-479876582        │
│ …                 ┆ …                   │
│ S1-7293388        ┆ S2-442723188        │
│ S1-7293388        ┆ S2-157073701        │
│ S1-546142636      ┆ S2-487600131        │
│ S1-546142636      ┆ S2-582477216        │
│ S1-546142636      ┆ S2-392804085        │
└───────────────────┴─────────────────────┘


In [39]:
missed_details = (
    exact_name_missed
    .join(
        s1.select([
            pl.col("entity_id").alias("source1_entity_id"),
            pl.col("name_norm").alias("s1_name"),
            pl.col("name_translit").alias("s1_translit"),
            pl.col("address_norm").alias("s1_address"),
            pl.col("country_norm").alias("s1_country"),
        ]),
        on="source1_entity_id",
        how="left",
    )
    .join(
        s2.select([
            pl.col("entity_id").alias("candidate_entity_id"),
            pl.col("name_norm").alias("s2_name"),
            pl.col("name_translit").alias("s2_translit"),
            pl.col("address_norm").alias("s2_address"),
            pl.col("country_norm").alias("s2_country"),
        ]),
        on="candidate_entity_id",
        how="left",
    )
)

print(
    missed_details.select([
        "source1_entity_id",
        "candidate_entity_id",
        "s1_name",
        "s2_name",
        "s1_translit",
        "s2_translit",
        "s1_country",
        "s2_country",
    ]).head(20)
)

shape: (20, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ source1_en ┆ candidate_ ┆ s1_name    ┆ s2_name   ┆ s1_transl ┆ s2_transl ┆ s1_countr ┆ s2_countr │
│ tity_id    ┆ entity_id  ┆ ---        ┆ ---       ┆ it        ┆ it        ┆ y         ┆ y         │
│ ---        ┆ ---        ┆ str        ┆ str       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆            ┆           ┆ str       ┆ str       ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ S1-965667  ┆ S2-6811933 ┆ maure      ┆ maure     ┆ maure     ┆ maure     ┆ us        ┆ us        │
│            ┆ 10         ┆ williams   ┆ wilblims  ┆ williams  ┆ wilblims  ┆           ┆           │
│            ┆            ┆ colombier  ┆ colombier ┆ colombier ┆ colombier ┆           ┆           │
│            ┆            ┆ inc        ┆ inc       ┆ inc       ┆ inc       ┆

In [40]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.5


In [41]:
DUCKDB_PATH = WORK_DIR / "entity_resolution.duckdb"

con = duckdb.connect(str(DUCKDB_PATH))

con.execute("""
    SET memory_limit='2GB';
""")

con.execute(f"""
    SET temp_directory='{WORK_DIR / "duckdb_temp"}';
""")

print("Database:", DUCKDB_PATH)
print("DuckDB configured.")

Database: dataset\work\entity_resolution.duckdb
DuckDB configured.


In [42]:
con.execute(f"""
    CREATE OR REPLACE VIEW train_s1 AS
    SELECT *
    FROM read_parquet('{NORM_TRAIN_S1}');
""")

con.execute(f"""
    CREATE OR REPLACE VIEW train_s2 AS
    SELECT *
    FROM read_parquet('{NORM_TRAIN_S2}');
""")

In [43]:
print(
    con.execute("""
        SELECT COUNT(*) FROM train_s1;
    """).fetchone()
)

print(
    con.execute("""
        SELECT COUNT(*) FROM train_s2;
    """).fetchone()
)

(2206821,)
(5034616,)


In [44]:
#37
LEGAL_SUFFIX_REGEX = (
    r'\b(private|pvt|limited|ltd|llp|inc|incorporated|'
    r'corporation|corp|company|co)\b'
)

In [45]:
con.execute(f"""
    CREATE OR REPLACE VIEW s2_canonical AS
    SELECT
        entity_id,
        name_norm,
        country_norm,
        regexp_replace(
            trim(
                regexp_replace(
                    name_norm,
                    '{LEGAL_SUFFIX_REGEX}',
                    ' ',
                    'gi'
                )
            ),
            '\\s+',
            ' ',
            'g'
        ) AS name_without_suffix
    FROM train_s2;
""")

In [46]:
import pandas as pd

In [47]:
print(
    con.execute("""
        SELECT
            name_norm,
            name_without_suffix
        FROM s2_canonical
        WHERE name_norm != ''
        LIMIT 20;
    """).fetchdf()
)

                                            name_norm  \
0                     राम मार्केटिंग प्राइवेट लिमिटेड   
1                           holloway peak inc seafood   
2                            आदित्य प्रॉपर्टीज एलएलपी   
3                                          summit inc   
4                        delta tetlecommunication inc   
5                                      lee and lawson   
6   shivshakti vidyalaya vidyalaya overseas corpor...   
7                          shree infracon private ltd   
8                        chavira platinum chimera llc   
9             foundation excel agency private limited   
10                                   heassociates com   
11                   सन कंस्ट्रक्शंस प्राइवेट लिमिटेड   
12         global developers medicals private limited   
13                       asset building coalition llc   
14                  olszewski holding company llc llc   
15                                   candy s services   
16                            र

In [48]:
con.execute(f"""
    CREATE OR REPLACE VIEW s2_canonical AS
    SELECT
        entity_id,
        name_norm,
        country_norm,

        trim(
            regexp_replace(
                regexp_replace(
                    name_norm,
                    '{LEGAL_SUFFIX_REGEX}',
                    ' ',
                    'gi'
                ),
                '\\s+',
                ' ',
                'g'
            )
        ) AS canonical_name

    FROM train_s2;
""")

In [49]:
con.execute(f"""
    CREATE OR REPLACE VIEW s1_canonical AS
    SELECT
        entity_id,
        name_norm,
        country_norm,

        trim(
            regexp_replace(
                regexp_replace(
                    name_norm,
                    '{LEGAL_SUFFIX_REGEX}',
                    ' ',
                    'gi'
                ),
                '\\s+',
                ' ',
                'g'
            )
        ) AS canonical_name

    FROM train_s1;
""")

In [50]:
print(
    con.execute("""
        SELECT
            name_norm,
            canonical_name
        FROM s2_canonical
        WHERE canonical_name != ''
        LIMIT 30;
    """).fetchdf()
)

                                            name_norm  \
0                     राम मार्केटिंग प्राइवेट लिमिटेड   
1                           holloway peak inc seafood   
2                            आदित्य प्रॉपर्टीज एलएलपी   
3                                          summit inc   
4                        delta tetlecommunication inc   
5                                      lee and lawson   
6   shivshakti vidyalaya vidyalaya overseas corpor...   
7                          shree infracon private ltd   
8                        chavira platinum chimera llc   
9             foundation excel agency private limited   
10                                   heassociates com   
11                   सन कंस्ट्रक्शंस प्राइवेट लिमिटेड   
12         global developers medicals private limited   
13                       asset building coalition llc   
14                  olszewski holding company llc llc   
15                                   candy s services   
16                            र

In [51]:
print(
    con.execute("""
        SELECT
            name_norm,
            canonical_name
        FROM s2_canonical
        WHERE name_norm != ''
        LIMIT 30;
    """).fetchdf().to_string(index=False)
)

                                                             name_norm                                             canonical_name
                                       राम मार्केटिंग प्राइवेट लिमिटेड                            राम मार्केटिंग प्राइवेट लिमिटेड
                                             holloway peak inc seafood                                      holloway peak seafood
                                              आदित्य प्रॉपर्टीज एलएलपी                                   आदित्य प्रॉपर्टीज एलएलपी
                                                            summit inc                                                     summit
                                          delta tetlecommunication inc                                   delta tetlecommunication
                                                        lee and lawson                                             lee and lawson
shivshakti vidyalaya vidyalaya overseas corporation www shivshakti com shivshakti vidyalay

In [52]:
canonical_frequency = con.execute("""
    SELECT
        canonical_name,
        COUNT(*) AS posting_count
    FROM s2_canonical
    WHERE canonical_name != ''
    GROUP BY canonical_name
    ORDER BY posting_count DESC
""").fetchdf()

print(canonical_frequency.head(20))

         canonical_name  posting_count
0      physical therapy            548
1          primary care            529
2         womens health            503
3              meridian            502
4     internal medicine            491
5     behavioral health            488
6      pediatric dental            482
7           urgent care            481
8   pediatric dentistry            471
9       ear nose throat            385
10           foot ankle            384
11        family center            360
12    earnosethroat com            355
13      mumbai services            323
14          main street            322
15             the dent            319
16        vision center            294
17        straight edge            291
18        commission on            287
19               helios            286


In [53]:
print(
    canonical_frequency["posting_count"].describe()
)

count    3.558335e+06
mean     1.414877e+00
std      3.126303e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.480000e+02
Name: posting_count, dtype: float64


In [54]:

MAX_CANONICAL_POSTINGS = 100
CANONICAL_CANDIDATES = CANDIDATE_DIR / "canonical_candidates.parquet"

con.execute(f"""
    COPY (
        WITH canonical_counts AS (
            SELECT
                canonical_name,
                COUNT(*) AS posting_count
            FROM s2_canonical
            WHERE canonical_name != ''
            GROUP BY canonical_name
        ),

        selective_s2 AS (
            SELECT
                s.entity_id AS candidate_entity_id,
                s.canonical_name
            FROM s2_canonical s
            INNER JOIN canonical_counts c
                ON s.canonical_name = c.canonical_name
            WHERE s.canonical_name != ''
              AND c.posting_count <= {MAX_CANONICAL_POSTINGS}
        )

        SELECT DISTINCT
            s1.entity_id AS source1_entity_id,
            s2.candidate_entity_id
        FROM s1_canonical s1
        INNER JOIN selective_s2 s2
            ON s1.canonical_name = s2.canonical_name
    )
    TO '{CANONICAL_CANDIDATES}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

In [55]:
canonical_count = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{CANONICAL_CANDIDATES}');
""").fetchone()[0]

print(
    f"Canonical candidate pairs: {canonical_count:,}"
)

Canonical candidate pairs: 16,045,598


In [56]:
con.register("gt_s2_df", gt_s2)

con.execute("""
    CREATE OR REPLACE TEMP VIEW ground_truth_s2 AS
    SELECT
        source1_entity_id,
        candidate_entity_id
    FROM gt_s2_df;
""")

In [57]:
canonical_recovered = con.execute(f"""
    SELECT COUNT(*)
    FROM ground_truth_s2 gt
    INNER JOIN read_parquet('{CANONICAL_CANDIDATES}') c
        ON gt.source1_entity_id = c.source1_entity_id
       AND gt.candidate_entity_id = c.candidate_entity_id;
""").fetchone()[0]

canonical_recall = canonical_recovered / len(gt_s2)

print(f"True pairs:       {len(gt_s2):,}")
print(f"Recovered pairs:  {canonical_recovered:,}")
print(f"Recall:           {canonical_recall:.4%}")

True pairs:       3,693,619
Recovered pairs:  1,374,967
Recall:           37.2255%


In [58]:
canonical_size_stats = con.execute(f"""
    SELECT
        AVG(candidate_count) AS mean,
        MEDIAN(candidate_count) AS median,
        QUANTILE_CONT(candidate_count, 0.95) AS p95,
        MAX(candidate_count) AS max
    FROM (
        SELECT
            source1_entity_id,
            COUNT(*) AS candidate_count
        FROM read_parquet('{CANONICAL_CANDIDATES}')
        GROUP BY source1_entity_id
    )
""").fetchone()

print(f"Mean:   {canonical_size_stats[0]:.2f}")
print(f"Median: {canonical_size_stats[1]:.2f}")
print(f"P95:    {canonical_size_stats[2]:.2f}")
print(f"Max:    {canonical_size_stats[3]:,}")

Mean:   10.53
Median: 2.00
P95:    48.00
Max:    100


In [59]:
coverage = con.execute(f"""
    SELECT
        COUNT(DISTINCT s1.entity_id) AS total_s1,
        COUNT(DISTINCT c.source1_entity_id) AS covered_s1
    FROM s1_canonical s1
    LEFT JOIN read_parquet('{CANONICAL_CANDIDATES}') c
        ON s1.entity_id = c.source1_entity_id
""").fetchone()

print(f"Total S1:    {coverage[0]:,}")
print(f"Covered S1:  {coverage[1]:,}")
print(f"Coverage:    {coverage[1] / coverage[0]:.4%}")

Total S1:    2,206,821
Covered S1:  1,523,794
Coverage:    69.0493%


In [60]:
con.execute("""
    CREATE OR REPLACE VIEW s2_char_signatures AS
    SELECT
        entity_id,
        name_norm,
        country_norm,

        LEFT(name_norm, 8) AS char_prefix8,

        RIGHT(name_norm, 8) AS char_suffix8,

        LENGTH(name_norm) AS name_length

    FROM train_s2
    WHERE name_norm != '';
""")

con.execute("""
    CREATE OR REPLACE VIEW s1_char_signatures AS
    SELECT
        entity_id,
        name_norm,
        country_norm,

        LEFT(name_norm, 8) AS char_prefix8,

        RIGHT(name_norm, 8) AS char_suffix8,

        LENGTH(name_norm) AS name_length

    FROM train_s1
    WHERE name_norm != '';
""")

In [61]:

#48
char_prefix_frequency = con.execute("""
    SELECT
        char_prefix8,
        COUNT(*) AS posting_count
    FROM s2_char_signatures
    WHERE char_prefix8 != ''
    GROUP BY char_prefix8
    ORDER BY posting_count DESC
""").fetchdf()

print(char_prefix_frequency.head(20).to_string(index=False))
print(
    char_prefix_frequency["posting_count"].describe()
)

char_prefix8  posting_count
    private           38676
    pediatri          30691
    limited           15083
    services          12862
    physical          10452
    cardiolo          10330
    dermatol          10193
    behavior          10059
    chiropra           9982
    primary            9894
    surgical           9859
    orthoped           9726
    oncology           9717
    internal           9646
    regional           9100
    urology            9045
    womens h           8940
    ear nose           8875
    urgent c           8796
    premier            8568
count    1.005588e+06
mean     5.006639e+00
std      9.089253e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      3.867600e+04
Name: posting_count, dtype: float64


In [62]:
char_suffix_frequency = con.execute("""
    SELECT
        char_suffix8,
        COUNT(*) AS posting_count
    FROM s2_char_signatures
    WHERE char_suffix8 != ''
    GROUP BY char_suffix8
    ORDER BY posting_count DESC
""").fetchdf()

print(char_suffix_frequency.head(20).to_string(index=False))

char_suffix8  posting_count
     limited         422625
     लिमिटेड         190091
     private         119112
     pvt ltd          99403
    services          98392
    partners          91535
    vate ltd          87955
    holdings          60956
    poration          56192
     service          45508
    c center          38581
     प्रा लि          35416
    d center          35298
    లిమిటెడ్          33008
    ಲಿಮಿಟೆಡ್          31320
    லிமிடெட்          28346
     লিমিটেড          25814
    sociates          24038
    rporated          23694
     company          23264


In [63]:
con.execute("""
    CREATE OR REPLACE VIEW s2_char_blocks AS
    SELECT
        entity_id,
        name_norm,
        country_norm,
        LEFT(name_norm, 8) AS char_prefix8,
        CAST(FLOOR(LENGTH(name_norm) / 5) AS INTEGER) AS length_bucket
    FROM train_s2
    WHERE name_norm != '';
""")

con.execute("""
    CREATE OR REPLACE VIEW s1_char_blocks AS
    SELECT
        entity_id,
        name_norm,
        country_norm,
        LEFT(name_norm, 8) AS char_prefix8,
        CAST(FLOOR(LENGTH(name_norm) / 5) AS INTEGER) AS length_bucket
    FROM train_s1
    WHERE name_norm != '';
""")

In [64]:
char_block_frequency = con.execute("""
    SELECT
        char_prefix8,
        country_norm,
        length_bucket,
        COUNT(*) AS posting_count
    FROM s2_char_blocks
    GROUP BY
        char_prefix8,
        country_norm,
        length_bucket
    ORDER BY posting_count DESC
""").fetchdf()

print(
    char_block_frequency.head(30).to_string(index=False)
)
print(
    char_block_frequency["posting_count"].describe()
)

char_prefix8 country_norm  length_bucket  posting_count
    private         india              6          12157
    private         india              5          11255
    private         india              7           6764
    pediatri           us              6           6404
    pediatri           us              5           6245
    pediatri           us              7           5224
    limited         india              5           4312
    pediatri           us              4           3945
    limited         india              6           3705
    private         india              8           3579
    pediatri           us              8           3260
    private         india              4           3191
    limited         india              4           3159
    pediatri           us              3           2807
    dermatol           us              4           2495
    physical           us              5           2450
    cardiolo           us              5        

In [65]:
con.execute("""
    CREATE OR REPLACE VIEW s2_char_blocks AS
    SELECT
        entity_id,
        name_norm,
        country_norm,
        LEFT(name_norm, 8) AS char_prefix8,
        RIGHT(name_norm, 6) AS char_suffix6,
        CAST(FLOOR(LENGTH(name_norm) / 5) AS INTEGER) AS length_bucket
    FROM train_s2
    WHERE name_norm != '';
""")

con.execute("""
    CREATE OR REPLACE VIEW s1_char_blocks AS
    SELECT
        entity_id,
        name_norm,
        country_norm,
        LEFT(name_norm, 8) AS char_prefix8,
        RIGHT(name_norm, 6) AS char_suffix6,
        CAST(FLOOR(LENGTH(name_norm) / 5) AS INTEGER) AS length_bucket
    FROM train_s1
    WHERE name_norm != '';
""")

In [66]:
char_block_frequency = con.execute("""
    SELECT
        char_prefix8,
        char_suffix6,
        country_norm,
        length_bucket,
        COUNT(*) AS posting_count
    FROM s2_char_blocks
    GROUP BY
        char_prefix8,
        char_suffix6,
        country_norm,
        length_bucket
    ORDER BY posting_count DESC
""").fetchdf()

print(char_block_frequency.head(30).to_string(index=False))
print("\nPosting statistics:")
print(char_block_frequency["posting_count"].describe())

char_prefix8 char_suffix6 country_norm  length_bucket  posting_count
    private        imited        india              6           8791
    private        imited        india              5           6285
    private        imited        india              7           5136
    private        imited        india              8           2938
    लक्ष्मी        िमिटेड        india              6           1320
    लक्ष्मी        िमिटेड        india              7           1139
    new delh       imited        india              6           1054
    private        imited        india              9           1024
    limited        center        india              5            978
    फॉर्च्यू       िमिटेड        india              7            826
    यूनिवर्स       िमिटेड        india              7            812
    इंटरनेशन       िमिटेड        india              7            803
    services       imited        india              7            794
    limited        rvices        i

In [67]:
test_tokens = con.execute("""
SELECT
    name_norm,
    string_split(name_norm, ' ') AS tokens
FROM train_s2
WHERE name_norm != ''
LIMIT 10
""").fetchdf()

print(test_tokens.to_string(index=False))

                                                             name_norm                                                                          tokens
                                       राम मार्केटिंग प्राइवेट लिमिटेड                                            [राम, मार्केटिंग, प्राइवेट, लिमिटेड]
                                             holloway peak inc seafood                                                  [holloway, peak, inc, seafood]
                                              आदित्य प्रॉपर्टीज एलएलपी                                                    [आदित्य, प्रॉपर्टीज, एलएलपी]
                                                            summit inc                                                                   [summit, inc]
                                          delta tetlecommunication inc                                                [delta, tetlecommunication, inc]
                                                        lee and lawson                        

In [68]:
token_test = con.execute("""
SELECT
    name_norm,

    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x -> x NOT IN (
                    'private', 'pvt', 'limited', 'ltd',
                    'llp', 'inc', 'incorporated',
                    'corporation', 'corp', 'company', 'co'
                )
                AND x != ''
            )
        ),
        ' '
    ) AS token_signature

FROM train_s2
WHERE name_norm != ''
LIMIT 20
""").fetchdf()

print(token_test.to_string(index=False))

                                                             name_norm                                            token_signature
                                       राम मार्केटिंग प्राइवेट लिमिटेड                            प्राइवेट मार्केटिंग राम लिमिटेड
                                             holloway peak inc seafood                                      holloway peak seafood
                                              आदित्य प्रॉपर्टीज एलएलपी                                   आदित्य एलएलपी प्रॉपर्टीज
                                                            summit inc                                                     summit
                                          delta tetlecommunication inc                                   delta tetlecommunication
                                                        lee and lawson                                             and lawson lee
shivshakti vidyalaya vidyalaya overseas corporation www shivshakti com com overseas shivsh

In [69]:
token_frequency = con.execute("""
SELECT
    token,
    COUNT(*) AS posting_count
FROM (
    SELECT
        entity_id,
        UNNEST(string_split(name_norm, ' ')) AS token
    FROM train_s2
    WHERE name_norm != ''
)
WHERE token != ''
GROUP BY token
ORDER BY posting_count DESC
LIMIT 100
""").fetchdf()

print(token_frequency.to_string(index=False))

        token  posting_count
          llc         532379
      limited         532213
      private         532072
          inc         403240
          ltd         401133
          com         202135
       center         196536
          pvt         190499
      लिमिटेड         190091
            l         178692
     प्राइवेट         157955
     partners         155487
     services         147339
         corp         142833
        group         141458
           co         140870
            s         139849
          and         118901
            c         116706
        india         114478
     holdings          97734
           of          89949
         care          89801
   associates          74001
          llp          65179
  corporation          63182
            p          57517
      service          55790
            d          51913
       health          51747
  enterprises          49677
   industries          48349
           lp          48301
       clinic 

In [70]:
LEGAL_TOKENS = [
    # English
    "private", "pvt", "limited", "ltd", "llc", "llp",
    "inc", "incorporated", "corp", "corporation",
    "company", "co", "lp", "pllc",

    # Hindi / Devanagari
    "प्राइवेट", "लिमिटेड", "प्रा", "लि",

    # Telugu
    "ప్రైవేట్", "లిమిటెడ్",

    # Kannada
    "ಪ್ರೈವೇಟ್", "ಲಿಮಿಟೆಡ್",

    # Tamil
    "பிரைவேட்", "லிமிடெட்",

    # Bengali
    "প্রাইভেট", "লিমিটেড",

    # Gujarati
    "પ્રાઇવેટ", "લિમિટેડ",
]

In [71]:
token_test = con.execute("""
SELECT
    name_norm,

    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x -> x NOT IN (
                    'private', 'pvt', 'limited', 'ltd',
                    'llc', 'llp', 'inc', 'incorporated',
                    'corp', 'corporation', 'company', 'co',
                    'lp', 'pllc',
                    'प्राइवेट', 'लिमिटेड', 'प्रा', 'लि',
                    'ప్రైవేట్', 'లిమిటెడ్',
                    'ಪ್ರೈವೇಟ್', 'ಲಿಮಿಟೆಡ್',
                    'பிரைவேட்', 'லிமிடெட்',
                    'প্রাইভেট', 'লিমিটেড',
                    'પ્રાઇવેટ', 'લિમિટેડ'
                )
                AND x != ''
            )
        ),
        ' '
    ) AS token_signature

FROM train_s2
WHERE name_norm != ''
LIMIT 30
""").fetchdf()

print(token_test.to_string(index=False))

                                                             name_norm                                            token_signature
                                       राम मार्केटिंग प्राइवेट लिमिटेड                                             मार्केटिंग राम
                                             holloway peak inc seafood                                      holloway peak seafood
                                              आदित्य प्रॉपर्टीज एलएलपी                                   आदित्य एलएलपी प्रॉपर्टीज
                                                            summit inc                                                     summit
                                          delta tetlecommunication inc                                   delta tetlecommunication
                                                        lee and lawson                                             and lawson lee
shivshakti vidyalaya vidyalaya overseas corporation www shivshakti com com overseas shivsh

In [72]:
LEGAL_TOKEN_SQL = """
'private', 'pvt', 'limited', 'ltd',
'llc', 'llp', 'inc', 'incorporated',
'corp', 'corporation', 'company', 'co',
'lp', 'pllc',
'प्राइवेट', 'लिमिटेड', 'प्रा', 'लि',
'ప్రైవేట్', 'లిమిటెడ్',
'ಪ್ರೈವೇಟ್', 'ಲಿಮಿಟೆಡ್',
'பிரைவேட்', 'லிமிடெட்',
'প্রাইভেট', 'লিমিটেড',
'પ્રાઇવેટ', 'લિમિટેડ'
"""

con.execute(f"""
CREATE OR REPLACE VIEW s2_token_signature AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x -> x NOT IN ({LEGAL_TOKEN_SQL})
                     AND x != ''
            )
        ),
        ' '
    ) AS token_signature
FROM train_s2
WHERE name_norm != '';
""")

con.execute(f"""
CREATE OR REPLACE VIEW s1_token_signature AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x -> x NOT IN ({LEGAL_TOKEN_SQL})
                     AND x != ''
            )
        ),
        ' '
    ) AS token_signature
FROM train_s1
WHERE name_norm != '';
""")

print("Token signature views created.")

Token signature views created.


In [73]:
token_examples = con.execute("""
SELECT
    name_norm,
    token_signature
FROM s2_token_signature
WHERE name_norm != ''
LIMIT 30
""").fetchdf()

print(token_examples.to_string(index=False))

                                                             name_norm                                            token_signature
                                       राम मार्केटिंग प्राइवेट लिमिटेड                                             मार्केटिंग राम
                                             holloway peak inc seafood                                      holloway peak seafood
                                              आदित्य प्रॉपर्टीज एलएलपी                                   आदित्य एलएलपी प्रॉपर्टीज
                                                            summit inc                                                     summit
                                          delta tetlecommunication inc                                   delta tetlecommunication
                                                        lee and lawson                                             and lawson lee
shivshakti vidyalaya vidyalaya overseas corporation www shivshakti com com overseas shivsh

In [74]:
token_frequency = con.execute("""
SELECT
    token_signature,
    COUNT(*) AS posting_count
FROM s2_token_signature
WHERE token_signature != ''
GROUP BY token_signature
ORDER BY posting_count DESC
""").fetchdf()

print(token_frequency.head(30).to_string(index=False))

print("\nPosting statistics:")
print(token_frequency["posting_count"].describe())

    token_signature  posting_count
           meridian            746
   physical therapy            567
       care primary            553
      health womens            522
  internal medicine            512
  behavioral health            507
        care urgent            503
   dental pediatric            497
      center family            492
dentistry pediatric            486
               lynx            416
              cedar            415
    ear nose throat            411
             summit            411
             anchor            410
             helios            407
           sapphire            405
         ankle foot            404
            cascade            402
             falcon            399
            halcyon            394
           sterling            391
           pinnacle            389
      center vision            385
            juniper            381
              ember            379
           ironclad            379
            citadel 

In [75]:
MAX_TOKEN_POSTINGS = 100

TOKEN_CANDIDATES = CANDIDATE_DIR / "token_candidates.parquet"

token_candidate_sql = f"""
COPY (
    WITH signature_counts AS (
        SELECT
            token_signature,
            COUNT(*) AS posting_count
        FROM s2_token_signature
        WHERE token_signature != ''
        GROUP BY token_signature
    ),

    selective_s2 AS (
        SELECT
            s.entity_id AS candidate_entity_id,
            s.token_signature,
            s.country_norm
        FROM s2_token_signature s
        INNER JOIN signature_counts c
            ON s.token_signature = c.token_signature
        WHERE s.token_signature != ''
          AND c.posting_count <= {MAX_TOKEN_POSTINGS}
    )

    SELECT DISTINCT
        s1.entity_id AS source1_entity_id,
        s2.candidate_entity_id
    FROM s1_token_signature s1
    INNER JOIN selective_s2 s2
        ON s1.token_signature = s2.token_signature
       AND (
            s1.country_norm = s2.country_norm
            OR s1.country_norm = ''
            OR s2.country_norm = ''
       )
)
TO '{TOKEN_CANDIDATES}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(token_candidate_sql)

print(f"Written: {TOKEN_CANDIDATES}")

Written: dataset\work\candidates\token_candidates.parquet


In [76]:
token_candidate_stats = con.execute(f"""
SELECT
    COUNT(*) AS total_candidates,
    COUNT(DISTINCT source1_entity_id) AS covered_s1,
    AVG(candidate_count) AS mean_candidates,
    MEDIAN(candidate_count) AS median_candidates,
    QUANTILE_CONT(candidate_count, 0.95) AS p95_candidates,
    MAX(candidate_count) AS max_candidates
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS candidate_count
    FROM read_parquet('{TOKEN_CANDIDATES}')
    GROUP BY source1_entity_id
)
""").fetchone()

print(token_candidate_stats)

token_recall = con.execute(f"""
SELECT
    COUNT(*) AS true_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    COUNT(c.candidate_entity_id) * 1.0 / COUNT(*) AS recall
FROM gt_s2 g
LEFT JOIN read_parquet('{TOKEN_CANDIDATES}') c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchone()

print(token_recall)

(1656513, 1656513, 10.381943274818852, 3.0, 51.0, 100)
(3693619, 1622603, 0.43929896397002505)


In [77]:
COMBINED_CANDIDATES = CANDIDATE_DIR / "combined_name_candidates.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM (
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_parquet('{CANONICAL_CANDIDATES}')

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_parquet('{TOKEN_CANDIDATES}')
    )
)
TO '{COMBINED_CANDIDATES}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Written: {COMBINED_CANDIDATES}")

Written: dataset\work\candidates\combined_name_candidates.parquet


In [78]:
combined_recall = con.execute(f"""
SELECT
    COUNT(*) AS true_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    COUNT(c.candidate_entity_id) * 1.0 / COUNT(*) AS recall
FROM gt_s2 g
LEFT JOIN read_parquet('{COMBINED_CANDIDATES}') c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchone()

print(combined_recall)

(3693619, 1641913, 0.444526898957364)


In [79]:
con.execute("""
CREATE OR REPLACE VIEW s2_address_numbers AS
SELECT
    entity_id,
    country_norm,
    address_norm,
    regexp_extract_all(address_norm, '[0-9]+') AS address_numbers
FROM train_s2
WHERE address_norm != '';
""")

con.execute("""
CREATE OR REPLACE VIEW s1_address_numbers AS
SELECT
    entity_id,
    country_norm,
    address_norm,
    regexp_extract_all(address_norm, '[0-9]+') AS address_numbers
FROM train_s1
WHERE address_norm != '';
""")

In [80]:
address_examples = con.execute("""
SELECT
    address_norm,
    address_numbers
FROM s2_address_numbers
WHERE address_norm != ''
LIMIT 30
""").fetchdf()

print(address_examples.to_string(index=False))

                                                                                               address_norm     address_numbers
                                                                    kh no 570 13 new delhi west delhi delhi           [570, 13]
                                                                                    105 elm st morganton nc               [105]
                                                              g 3 571 gulmohar colony bhopal madhya pradesh            [3, 571]
                                                                        greensboro nc 19 1 2 stardust trail          [19, 1, 2]
                                                                              914 pierpont ave cleveland oh               [914]
                                                                      1702 pine avenue city of menomonie wi              [1702]
                                                                   h no 204 c road hoshiarpur punjab pun

In [81]:
address_number_frequency = con.execute("""
SELECT
    number,
    COUNT(*) AS posting_count
FROM (
    SELECT
        entity_id,
        UNNEST(address_numbers) AS number
    FROM s2_address_numbers
)
WHERE number != ''
GROUP BY number
ORDER BY posting_count DESC
LIMIT 100
""").fetchdf()

print(address_number_frequency.to_string(index=False))

number  posting_count
     1         411608
     2         345711
     3         224813
     4         154525
     5         126316
     6         109113
     7          96221
     8          95555
     9          79918
    10          71517
    11          65435
    12          61943
    14          51989
    15          48950
    13          48469
    16          47438
    17          42142
    18          39821
    24          39063
    19          38474
    20          35412
    22          35076
    21          33522
    23          33199
    25          29801
    27          26476
    30          26396
    26          25754
    28          25174
    29          23862
    31          23459
    32          22743
    33          21516
    40          20923
    35          20879
    36          20374
    34          20369
   101          19333
    37          18951
    39          18218
    41          18101
    38          17742
    50          17281
    42          16984
    44    

In [82]:
number_stats = con.execute("""
WITH number_counts AS (
    SELECT
        number,
        COUNT(*) AS posting_count
    FROM (
        SELECT
            entity_id,
            UNNEST(address_numbers) AS number
        FROM s2_address_numbers
    )
    WHERE number != ''
    GROUP BY number
)
SELECT
    COUNT(*) AS distinct_numbers,
    SUM(CASE WHEN posting_count <= 10 THEN 1 ELSE 0 END) AS numbers_le_10,
    SUM(CASE WHEN posting_count <= 25 THEN 1 ELSE 0 END) AS numbers_le_25,
    SUM(CASE WHEN posting_count <= 50 THEN 1 ELSE 0 END) AS numbers_le_50,
    SUM(CASE WHEN posting_count <= 100 THEN 1 ELSE 0 END) AS numbers_le_100,
    MAX(posting_count) AS max_posting
FROM number_counts
""").fetchone()

print(number_stats)
rare_numbers = con.execute("""
SELECT
    number,
    COUNT(*) AS posting_count
FROM (
    SELECT
        entity_id,
        UNNEST(address_numbers) AS number
    FROM s2_address_numbers
)
WHERE number != ''
GROUP BY number
HAVING COUNT(*) <= 10
ORDER BY posting_count DESC, number
LIMIT 50
""").fetchdf()

print(rare_numbers.to_string(index=False))

(72980, 52044, 59236, 63326, 67122, 411608)
 number  posting_count
 001041             10
 001054             10
 001059             10
 001065             10
 001085             10
 001136             10
 001143             10
 001151             10
 001174             10
 001222             10
 001227             10
 001233             10
 001240             10
 001245             10
 001254             10
 001260             10
 001280             10
 001304             10
 001327             10
0013581             10
 001379             10
 001390             10
 001407             10
 001417             10
 001425             10
 001447             10
 001459             10
 001508             10
 001519             10
 001524             10
 001549             10
 001564             10
 001598             10
 001626             10
 001629             10
0016326             10
 001703             10
 001712             10
 001730             10
 001731             10
 001817      

In [84]:
MAX_NUMBER_POSTINGS = 25

NUMBER_CANDIDATES = CANDIDATE_DIR / "address_number_candidates.parquet"

con.execute(f"""
COPY (
    WITH number_counts AS (
        SELECT
            number,
            COUNT(*) AS posting_count
        FROM (
            SELECT
                entity_id,
                UNNEST(address_numbers) AS number
            FROM s2_address_numbers
        )
        WHERE number != ''
        GROUP BY number
    ),

    selective_s2 AS (
        SELECT
            s.entity_id AS candidate_entity_id,
            s.country_norm,
            n.number
        FROM s2_address_numbers s
        CROSS JOIN UNNEST(s.address_numbers) AS n(number)
        INNER JOIN number_counts c
            ON n.number = c.number
        WHERE n.number != ''
          AND c.posting_count <= {MAX_NUMBER_POSTINGS}
    ),

    s1_numbers AS (
        SELECT
            s.entity_id AS source1_entity_id,
            s.country_norm,
            n.number
        FROM s1_address_numbers s
        CROSS JOIN UNNEST(s.address_numbers) AS n(number)
        WHERE n.number != ''
    )

    SELECT DISTINCT
        s1.source1_entity_id,
        s2.candidate_entity_id
    FROM s1_numbers s1
    INNER JOIN selective_s2 s2
        ON s1.number = s2.number
       AND (
            s1.country_norm = s2.country_norm
            OR s1.country_norm = ''
            OR s2.country_norm = ''
       )
)
TO '{NUMBER_CANDIDATES}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Written: {NUMBER_CANDIDATES}")

number_candidate_stats = con.execute(f"""
SELECT
    COUNT(*) AS total_candidates,
    COUNT(DISTINCT source1_entity_id) AS covered_s1,
    AVG(candidate_count) AS mean_candidates,
    MEDIAN(candidate_count) AS median_candidates,
    QUANTILE_CONT(candidate_count, 0.95) AS p95_candidates,
    MAX(candidate_count) AS max_candidates
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS candidate_count
    FROM read_parquet('{NUMBER_CANDIDATES}')
    GROUP BY source1_entity_id
)
""").fetchone()

print(number_candidate_stats)

number_recall = con.execute(f"""
SELECT
    COUNT(*) AS true_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    COUNT(c.candidate_entity_id) * 1.0 / COUNT(*) AS recall
FROM gt_s2 g
LEFT JOIN read_parquet('{NUMBER_CANDIDATES}') c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchone()

print(number_recall)

Written: dataset\work\candidates\address_number_candidates.parquet
(91956, 91956, 10.805722301970508, 10.0, 23.0, 48)
(3693619, 114283, 0.03094065738778147)


In [85]:
con.execute("""
CREATE OR REPLACE VIEW s2_address_tokens AS
SELECT
    entity_id,
    country_norm,
    address_norm,
    UNNEST(string_split(address_norm, ' ')) AS token
FROM train_s2
WHERE address_norm != '';
""")

con.execute("""
CREATE OR REPLACE VIEW s1_address_tokens AS
SELECT
    entity_id,
    country_norm,
    address_norm,
    UNNEST(string_split(address_norm, ' ')) AS token
FROM train_s1
WHERE address_norm != '';
""")

print("Address token views created.")

Address token views created.


In [86]:
address_token_frequency = con.execute("""
SELECT
    token,
    COUNT(DISTINCT entity_id) AS posting_count
FROM s2_address_tokens
WHERE token != ''
GROUP BY token
ORDER BY posting_count DESC
LIMIT 100
""").fetchdf()

print(address_token_frequency.to_string(index=False))

      token  posting_count
         no         916454
       road         669292
     street         337310
         st         333483
         rd         330958
      floor         319586
maharashtra         319026
       city         298419
         tx         292289
      nagar         286961
         dr         283537
          1         281619
          a         265437
      delhi         262478
          c         254378
          2         228481
         ny         225422
        ave         224070
      drive         215965
    pradesh         214708
        new         210472
         nc         207687
       west         200360
     avenue         193632
         oh         187361
          h         186398
          b         183599
       plot         178181
         il         165947
     mumbai         162530
          o         154052
       null         131794
         va         131283
         tn         131049
         ma         125903
         az         124432
 

In [87]:
con.execute("""
CREATE OR REPLACE VIEW s2_postal_codes AS
SELECT
    entity_id,
    country_norm,
    address_norm,
    regexp_extract_all(address_norm, '\\\\b[0-9]{5,6}\\\\b') AS postal_codes
FROM train_s2
WHERE address_norm != '';
""")

con.execute("""
CREATE OR REPLACE VIEW s1_postal_codes AS
SELECT
    entity_id,
    country_norm,
    address_norm,
    regexp_extract_all(address_norm, '\\\\b[0-9]{5,6}\\\\b') AS postal_codes
FROM train_s1
WHERE address_norm != '';
""")

In [88]:
postal_examples = con.execute("""
SELECT
    address_norm,
    postal_codes
FROM s2_postal_codes
WHERE len(postal_codes) > 0
LIMIT 30
""").fetchdf()

print(postal_examples.to_string(index=False))

Empty DataFrame
Columns: [address_norm, postal_codes]
Index: []


In [89]:
con.execute("""
CREATE OR REPLACE VIEW s2_name_char_blocks AS
SELECT
    entity_id,
    country_norm,
    token_signature,
    LEFT(token_signature, 5) AS prefix5,
    CASE
        WHEN LENGTH(token_signature) >= 10
        THEN SUBSTRING(
            token_signature,
            CAST(FLOOR(LENGTH(token_signature) / 2) - 2 AS BIGINT),
            5
        )
        ELSE token_signature
    END AS middle5
FROM s2_token_signature
WHERE token_signature != '';
""")

con.execute("""
CREATE OR REPLACE VIEW s1_name_char_blocks AS
SELECT
    entity_id,
    country_norm,
    token_signature,
    LEFT(token_signature, 5) AS prefix5,
    CASE
        WHEN LENGTH(token_signature) >= 10
        THEN SUBSTRING(
            token_signature,
            CAST(FLOOR(LENGTH(token_signature) / 2) - 2 AS BIGINT),
            5
        )
        ELSE token_signature
    END AS middle5
FROM s1_token_signature
WHERE token_signature != '';
""")

print("Character block views created.")

Character block views created.


In [90]:
char_block_frequency = con.execute("""
SELECT
    prefix5,
    middle5,
    country_norm,
    COUNT(*) AS posting_count
FROM s2_name_char_blocks
GROUP BY prefix5, middle5, country_norm
ORDER BY posting_count DESC
LIMIT 30
""").fetchdf()

print(char_block_frequency.to_string(index=False))

prefix5 middle5 country_norm  posting_count
  broth   hers         india           5822
  cente   nter         india           4600
  broth   thers        india           3504
  cente   nter            us           3400
  servi   ices         india           2579
  कंस्ट   ्रक्श        india           2541
  consu   ultan        india           2349
  इन्वे   ्टमें        india           2319
  found   ation        india           2189
  cardi   ology           us           2185
  const   uctio        india           2161
  partn   ners            us           2048
  enter   rises        india           1885
  टेक्न   ोलॉजी        india           1880
  const   ction        india           1876
  consu   tants        india           1871
  consu   ltant        india           1839
  इन्वे   टमेंट        india           1826
  consu   tancy        india           1802
  enter   prise        india           1747
  cardi   logy            us           1734
  servi   vices        india    

In [91]:
missed_examples = con.execute(f"""
SELECT
    g.source1_entity_id,
    g.candidate_entity_id AS true_entity_id,
    s1.name_norm AS source1_name,
    s2.name_norm AS true_name,
    s1.address_norm AS source1_address,
    s2.address_norm AS true_address,
    s1.country_norm AS country
FROM gt_s2 g
INNER JOIN train_s1 s1
    ON g.source1_entity_id = s1.entity_id
INNER JOIN train_s2 s2
    ON g.candidate_entity_id = s2.entity_id
WHERE NOT EXISTS (
    SELECT 1
    FROM read_parquet('{COMBINED_CANDIDATES}') c
    WHERE c.source1_entity_id = g.source1_entity_id
      AND c.candidate_entity_id = g.candidate_entity_id
)
USING SAMPLE 50
""").fetchdf()

print(missed_examples.to_string(index=False))

source1_entity_id true_entity_id                         source1_name                             true_name                                                                                    source1_address                                                                                        true_address country
     S1-292568175   S2-637094253                      burton controls                      burton cdonchrsl                                                               992 rippin run road greene county va                                                                992 rippin run road greene county va      us
     S1-523715639   S2-201587471              newton prairie sony llc                 newton prairie sy llc                                                                902 jones dairy road wake forest nc                                                                     jones dairy road wake forest nc      us
     S1-725814772   S2-976632400        shroyer escoto 

In [92]:
con.execute(f"""
CREATE OR REPLACE VIEW s2_translit_token_signature AS
SELECT
    entity_id,
    name_norm,
    name_translit,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_translit, ' '),
                x -> x NOT IN ({LEGAL_TOKEN_SQL})
                     AND x != ''
            )
        ),
        ' '
    ) AS translit_signature
FROM read_parquet('{NORM_TRAIN_S2}')
WHERE name_translit != '';
""")

con.execute(f"""
CREATE OR REPLACE VIEW s1_translit_token_signature AS
SELECT
    entity_id,
    name_norm,
    name_translit,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_translit, ' '),
                x -> x NOT IN ({LEGAL_TOKEN_SQL})
                     AND x != ''
            )
        ),
        ' '
    ) AS translit_signature
FROM read_parquet('{NORM_TRAIN_S1}')
WHERE name_translit != '';
""")

print("Transliteration token-signature views created.")

Transliteration token-signature views created.


In [93]:
translit_examples = con.execute("""
SELECT
    name_norm,
    name_translit,
    translit_signature
FROM s2_translit_token_signature
LIMIT 30
""").fetchdf()

print(translit_examples.to_string(index=False))

                                                             name_norm                                                          name_translit                                         translit_signature
                                       राम मार्केटिंग प्राइवेट लिमिटेड                                   raam maarkettiNg praaivett limittedd                       limittedd maarkettiNg praaivett raam
                                             holloway peak inc seafood                                              holloway peak inc seafood                                      holloway peak seafood
                                              आदित्य प्रॉपर्टीज एलएलपी                                             aadity proNprttiij elelpii                                 aadity elelpii proNprttiij
                                                            summit inc                                                             summit inc                                                     su

In [94]:
translit_frequency = con.execute("""
SELECT
    translit_signature,
    COUNT(*) AS posting_count
FROM s2_translit_token_signature
WHERE translit_signature != ''
GROUP BY translit_signature
ORDER BY posting_count DESC
LIMIT 30
""").fetchdf()

print(translit_frequency.to_string(index=False))

 translit_signature  posting_count
           meridian            788
   physical therapy            580
       care primary            577
      center family            543
      health womens            538
  internal medicine            526
  behavioral health            520
        care urgent            516
   dental pediatric            516
dentistry pediatric            500
              cedar            437
             summit            435
             anchor            432
             helios            430
            cascade            430
           sapphire            425
               lynx            423
    ear nose throat            419
         ankle foot            417
      center vision            415
            halcyon            414
             falcon            411
           sterling            404
           pinnacle            404
           ironclad            403
            juniper            398
              ember            395
              slate 

In [95]:
translit_missed_examples = con.execute(f"""
SELECT
    g.source1_entity_id,
    g.candidate_entity_id AS true_entity_id,

    s1.name_norm AS source1_name,
    s2.name_norm AS true_name,

    s1.translit_signature AS source1_translit,
    s2.translit_signature AS true_translit,

    CASE
        WHEN s1.translit_signature = s2.translit_signature
        THEN TRUE
        ELSE FALSE
    END AS translit_exact_match

FROM gt_s2 g

INNER JOIN s1_translit_token_signature s1
    ON g.source1_entity_id = s1.entity_id

INNER JOIN s2_translit_token_signature s2
    ON g.candidate_entity_id = s2.entity_id

WHERE NOT EXISTS (
    SELECT 1
    FROM read_parquet('{COMBINED_CANDIDATES}') c
    WHERE c.source1_entity_id = g.source1_entity_id
      AND c.candidate_entity_id = g.candidate_entity_id
)

USING SAMPLE 100
""").fetchdf()

print(translit_missed_examples.to_string(index=False))

source1_entity_id true_entity_id                         source1_name                                        true_name                     source1_translit                                       true_translit  translit_exact_match
     S1-233873368   S2-159912400 physical therapy physicians of tempe physical therapy physicians physicians of taahem of physical physicians tempe therapy    of physical physicians physicians taahem therapy                 False
     S1-769900880   S2-486204310               west montessori school                             west school services               montessori school west                                school services west                 False
     S1-910593486    S2-41111346          rapid ocean private limited                     rapid ocean piramvte limited                          ocean rapid                                ocean piramvte rapid                 False
     S1-832442914     S2-3762241                  schultz united bond           

In [96]:
translit_recovery = con.execute(f"""
SELECT
    COUNT(*) AS missed_true_pairs,

    SUM(
        CASE
            WHEN s1.translit_signature = s2.translit_signature
            THEN 1
            ELSE 0
        END
    ) AS translit_recovered,

    SUM(
        CASE
            WHEN s1.translit_signature = s2.translit_signature
            THEN 1
            ELSE 0
        END
    ) * 1.0 / COUNT(*) AS recovery_rate

FROM gt_s2 g

INNER JOIN s1_translit_token_signature s1
    ON g.source1_entity_id = s1.entity_id

INNER JOIN s2_translit_token_signature s2
    ON g.candidate_entity_id = s2.entity_id

WHERE NOT EXISTS (
    SELECT 1
    FROM read_parquet('{COMBINED_CANDIDATES}') c
    WHERE c.source1_entity_id = g.source1_entity_id
      AND c.candidate_entity_id = g.candidate_entity_id
)
""").fetchone()

print(translit_recovery)

(2051706, 245313, 0.11956537632584785)


In [97]:
MAX_TRANSLIT_POSTINGS = 100

TRANSLIT_CANDIDATES = CANDIDATE_DIR / "translit_candidates.parquet"

con.execute(f"""
COPY (
    WITH signature_counts AS (
        SELECT
            translit_signature,
            COUNT(*) AS posting_count
        FROM s2_translit_token_signature
        WHERE translit_signature != ''
        GROUP BY translit_signature
    ),

    selective_s2 AS (
        SELECT
            s.entity_id AS candidate_entity_id,
            s.translit_signature,
            s.country_norm
        FROM s2_translit_token_signature s
        INNER JOIN signature_counts c
            ON s.translit_signature = c.translit_signature
        WHERE s.translit_signature != ''
          AND c.posting_count <= {MAX_TRANSLIT_POSTINGS}
    )

    SELECT DISTINCT
        s1.entity_id AS source1_entity_id,
        s2.candidate_entity_id
    FROM s1_translit_token_signature s1
    INNER JOIN selective_s2 s2
        ON s1.translit_signature = s2.translit_signature
       AND (
            s1.country_norm = s2.country_norm
            OR s1.country_norm = ''
            OR s2.country_norm = ''
       )
)
TO '{TRANSLIT_CANDIDATES}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Written: {TRANSLIT_CANDIDATES}")

Written: dataset\work\candidates\translit_candidates.parquet


In [98]:
translit_candidate_stats = con.execute(f"""
SELECT
    COUNT(*) AS total_candidates,
    COUNT(DISTINCT source1_entity_id) AS covered_s1,
    AVG(candidate_count) AS mean_candidates,
    MEDIAN(candidate_count) AS median_candidates,
    QUANTILE_CONT(candidate_count, 0.95) AS p95_candidates,
    MAX(candidate_count) AS max_candidates
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS candidate_count
    FROM read_parquet('{TRANSLIT_CANDIDATES}')
    GROUP BY source1_entity_id
)
""").fetchone()

print(translit_candidate_stats)

(1697524, 1697524, 10.363631972213648, 3.0, 50.0, 100)


In [99]:
translit_recall = con.execute(f"""
SELECT
    COUNT(*) AS true_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    COUNT(c.candidate_entity_id) * 1.0 / COUNT(*) AS recall
FROM gt_s2 g
LEFT JOIN read_parquet('{TRANSLIT_CANDIDATES}') c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchone()

print(translit_recall)

(3693619, 1785956, 0.48352469488596417)


In [100]:
COMBINED_V2 = CANDIDATE_DIR / "combined_name_translit_candidates.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM (
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_parquet('{COMBINED_CANDIDATES}')

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_parquet('{TRANSLIT_CANDIDATES}')
    )
)
TO '{COMBINED_V2}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Written: {COMBINED_V2}")

Written: dataset\work\candidates\combined_name_translit_candidates.parquet


In [101]:
combined_v2_stats = con.execute(f"""
SELECT
    COUNT(*) AS total_candidates,
    COUNT(DISTINCT source1_entity_id) AS covered_s1,
    AVG(candidate_count) AS mean_candidates,
    MEDIAN(candidate_count) AS median_candidates,
    QUANTILE_CONT(candidate_count, 0.95) AS p95_candidates,
    MAX(candidate_count) AS max_candidates
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS candidate_count
    FROM read_parquet('{COMBINED_V2}')
    GROUP BY source1_entity_id
)
""").fetchone()

print(combined_v2_stats)

combined_v2_recall = con.execute(f"""
SELECT
    COUNT(*) AS true_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    COUNT(c.candidate_entity_id) * 1.0 / COUNT(*) AS recall
FROM gt_s2 g
LEFT JOIN read_parquet('{COMBINED_V2}') c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchone()

print(combined_v2_recall)

(1744000, 1744000, 11.952309059633027, 3.0, 60.0, 100)
(3693619, 1816235, 0.49172234602431925)


In [102]:
fuzzy_test = con.execute("""
SELECT
    jaro_winkler_similarity(
        'cutler ramirez',
        'cutlerramirezcom'
    ) AS jw_similarity,

    levenshtein(
        'collins lake and tedesco',
        'c0llins lake and tedesco'
    ) AS edit_distance
""").fetchone()

print(fuzzy_test)

(0.9482142857142858, 1)


In [103]:
missed_fuzzy_stats = con.execute(f"""
SELECT
    COUNT(*) AS missed_pairs,

    AVG(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        )
    ) AS avg_jw,

    MEDIAN(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        )
    ) AS median_jw,

    QUANTILE_CONT(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ), 0.10
    ) AS p10_jw,

    QUANTILE_CONT(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ), 0.25
    ) AS p25_jw,

    QUANTILE_CONT(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ), 0.50
    ) AS p50_jw,

    QUANTILE_CONT(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ), 0.75
    ) AS p75_jw,

    QUANTILE_CONT(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ), 0.90
    ) AS p90_jw

FROM gt_s2 g

INNER JOIN train_s1 s1
    ON g.source1_entity_id = s1.entity_id

INNER JOIN train_s2 s2
    ON g.candidate_entity_id = s2.entity_id

WHERE NOT EXISTS (
    SELECT 1
    FROM read_parquet('{COMBINED_V2}') c
    WHERE c.source1_entity_id = g.source1_entity_id
      AND c.candidate_entity_id = g.candidate_entity_id
)
""").fetchone()

print(missed_fuzzy_stats)

(1877384, 0.7865848397180881, 0.8937499999999999, 0.3713387241689128, 0.6969696969696969, 0.8937499999999999, 0.9421052631578947, 0.973517786561265)


In [104]:
low_similarity_examples = con.execute(f"""
SELECT
    s1.name_norm AS source1_name,
    s2.name_norm AS true_name,

    ROUND(
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ), 4
    ) AS jw_similarity

FROM gt_s2 g

INNER JOIN train_s1 s1
    ON g.source1_entity_id = s1.entity_id

INNER JOIN train_s2 s2
    ON g.candidate_entity_id = s2.entity_id

WHERE NOT EXISTS (
    SELECT 1
    FROM read_parquet('{COMBINED_V2}') c
    WHERE c.source1_entity_id = g.source1_entity_id
      AND c.candidate_entity_id = g.candidate_entity_id
)

ORDER BY jw_similarity ASC
LIMIT 50
""").fetchdf()

print(low_similarity_examples.to_string(index=False))

                 source1_name         true_name  jw_similarity
              galaxy products गैलेक्सी products            0.0
                        white       lumnylaorbi            0.0
                      fhs btc            qu0jax            0.0
                 hj p pvt ltd          fayebrix            0.0
                 dynamic tech      ડાયનેમિક ટેક            0.0
                 dynamic tech      डायनामिक टेक            0.0
             vsb rig of vasai           zetalum            0.0
                     keo pllc        vantageavi            0.0
                fortune power    फॉर्च्यून पावर            0.0
                 nexa capital            korlum            0.0
                         seas           novinex            0.0
                perfect power   பர்ஃபெக்ட் பவர்            0.0
                      gzk cdt           ciravio            0.0
                         onyx         umbrabe1o            0.0
               dossey college      ciraariariza        

In [105]:
FUZZY_SCORED_CANDIDATES = CANDIDATE_DIR / "fuzzy_scored_candidates.parquet"

con.execute(f"""
COPY (
    SELECT
        c.source1_entity_id,
        c.candidate_entity_id,

        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ) AS name_jw,

        levenshtein(
            s1.name_norm,
            s2.name_norm
        ) AS name_edit_distance,

        jaro_winkler_similarity(
            s1.address_norm,
            s2.address_norm
        ) AS address_jw

    FROM read_parquet('{COMBINED_V2}') c

    INNER JOIN train_s1 s1
        ON c.source1_entity_id = s1.entity_id

    INNER JOIN train_s2 s2
        ON c.candidate_entity_id = s2.entity_id
)
TO '{FUZZY_SCORED_CANDIDATES}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Written: {FUZZY_SCORED_CANDIDATES}")

Written: dataset\work\candidates\fuzzy_scored_candidates.parquet


In [106]:
score_separation = con.execute(f"""
SELECT
    CASE
        WHEN EXISTS (
            SELECT 1
            FROM gt_s2 g
            WHERE g.source1_entity_id = c.source1_entity_id
              AND g.candidate_entity_id = c.candidate_entity_id
        )
        THEN 'TRUE_MATCH'
        ELSE 'NON_MATCH'
    END AS label,

    COUNT(*) AS pairs,

    AVG(name_jw) AS avg_name_jw,
    MEDIAN(name_jw) AS median_name_jw,

    QUANTILE_CONT(name_jw, 0.10) AS p10_name_jw,
    QUANTILE_CONT(name_jw, 0.25) AS p25_name_jw,
    QUANTILE_CONT(name_jw, 0.75) AS p75_name_jw,
    QUANTILE_CONT(name_jw, 0.90) AS p90_name_jw

FROM read_parquet('{FUZZY_SCORED_CANDIDATES}') c
GROUP BY label
ORDER BY label
""").fetchdf()

print(score_separation.to_string(index=False))

     label    pairs  avg_name_jw  median_name_jw  p10_name_jw  p25_name_jw  p75_name_jw  p90_name_jw
 NON_MATCH 19028592     0.942594        0.957143     0.880878     0.922294          1.0          1.0
TRUE_MATCH  1816235     0.962038        0.973518     0.914286     0.950637          1.0          1.0


In [ ]:
# score_separation = con.execute(f"""
# SELECT
#     CASE
#         WHEN EXISTS (
#             SELECT 1
#             FROM gt_s2 g
#             WHERE g.source1_entity_id = c.source1_entity_id
#               AND g.candidate_entity_id = c.candidate_entity_id
#         )
#         THEN 'TRUE_MATCH'
#         ELSE 'NON_MATCH'
#     END AS label,

#     COUNT(*) AS pairs,

#     ROUND(AVG(name_jw), 6) AS avg_name_jw,
#     ROUND(AVG(address_jw), 6) AS avg_address_jw,

#     ROUND(MEDIAN(name_jw), 6) AS median_name_jw,
#     ROUND(MEDIAN(address_jw), 6) AS median_address_jw,

#     ROUND(QUANTILE_CONT(name_jw, 0.10), 6) AS p10_name_jw,
#     ROUND(QUANTILE_CONT(address_jw, 0.10), 6) AS p10_address_jw,

#     ROUND(QUANTILE_CONT(name_jw, 0.25), 6) AS p25_name_jw,
#     ROUND(QUANTILE_CONT(address_jw, 0.25), 6) AS p25_address_jw,

#     ROUND(QUANTILE_CONT(name_jw, 0.75), 6) AS p75_name_jw,
#     ROUND(QUANTILE_CONT(address_jw, 0.75), 6) AS p75_address_jw

# FROM read_parquet('{FUZZY_SCORED_CANDIDATES}') c

# GROUP BY 1
# ORDER BY 1
# """).pl()

# score_separation

OutOfMemoryException: Out of Memory Error: Allocation failure

In [107]:
LABELED_FUZZY = CANDIDATE_DIR / "fuzzy_scored_labeled.parquet"

con.execute(f"""
COPY (
    SELECT
        c.source1_entity_id,
        c.candidate_entity_id,
        c.name_jw,
        c.name_edit_distance,
        c.address_jw,
        CASE
            WHEN g.candidate_entity_id IS NOT NULL
            THEN 'TRUE_MATCH'
            ELSE 'NON_MATCH'
        END AS label
    FROM read_parquet('{FUZZY_SCORED_CANDIDATES}') c
    LEFT JOIN gt_s2 g
        ON c.source1_entity_id = g.source1_entity_id
       AND c.candidate_entity_id = g.candidate_entity_id
)
TO '{LABELED_FUZZY}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(LABELED_FUZZY)

dataset\work\candidates\fuzzy_scored_labeled.parquet


In [12]:
from pathlib import Path
import duckdb
import polars as pl

DATA_ROOT = Path(r"D:\business_entity_resolution")
WORK_DIR = DATA_ROOT / "dataset/work"
CANDIDATE_DIR = WORK_DIR / "candidates"

NORM_TRAIN_S1 = WORK_DIR / "normalized" / "train_source1.parquet"
NORM_TRAIN_S2 = WORK_DIR / "normalized" / "train_source2.parquet"

FUZZY_SCORED_CANDIDATES = CANDIDATE_DIR / "fuzzy_scored_candidates.parquet"
LABELED_FUZZY = CANDIDATE_DIR / "fuzzy_scored_labeled.parquet"

DUCKDB_PATH = WORK_DIR / "entity_resolution.duckdb"

con = duckdb.connect(str(DUCKDB_PATH))

con.execute("SET memory_limit='2GB';")
con.execute(f"SET temp_directory='{WORK_DIR / 'duckdb_temp'}';")

print("DuckDB restored")
print("Fuzzy candidates:", FUZZY_SCORED_CANDIDATES.exists())
print("Labeled fuzzy:", LABELED_FUZZY.exists())

DuckDB restored
Fuzzy candidates: True
Labeled fuzzy: True


In [13]:
con.execute("SHOW TABLES").pl()

name
str
"""s1_address_numbers"""
"""s1_address_tokens"""
"""s1_canonical"""
"""s1_char_blocks"""
"""s1_char_signatures"""
…
"""s2_postal_codes"""
"""s2_token_signature"""
"""s2_translit_token_signature"""


In [14]:
GROUND_TRUTH = r"D:\business_entity_resolution\dataset\train\train_ground_truth.tsv"

gt = pl.read_csv(
    GROUND_TRUTH,
    separator="\t"
)

gt_pairs = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .fill_null("")
        .str.split(",")
        .alias("matched_ids")
    )
    .explode("matched_ids")
    .rename({"matched_ids": "candidate_entity_id"})
    .filter(
        pl.col("candidate_entity_id").str.len_chars() > 0
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id"
    ])
)

gt_s2 = gt_pairs.filter(
    pl.col("candidate_entity_id").str.starts_with("S2-")
)

con.register("gt_s2", gt_s2)

print(gt_s2.shape)

C:\Users\KRISH\tmp\ipykernel_10340\3941382100.py:16: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("matched_ids")


(3693619, 2)


In [15]:
address_stats = con.execute(f"""
SELECT
    label,
    COUNT(*) AS pairs,
    ROUND(AVG(address_jw), 6) AS avg_address_jw
FROM read_parquet('{LABELED_FUZZY}')
GROUP BY label
ORDER BY label
""").pl()

address_stats

label,pairs,avg_address_jw
str,i64,f64
"""NON_MATCH""",19028592,0.558232
"""TRUE_MATCH""",1816235,0.817993


In [17]:
combined_stats = con.execute(f"""
SELECT
    label,
    COUNT(*) AS pairs,
    ROUND(
        AVG(0.35 * name_jw + 0.65 * address_jw),
        6
    ) AS avg_combined_score
FROM read_parquet('{LABELED_FUZZY}')
GROUP BY label
ORDER BY label
""").pl()

combined_stats

label,pairs,avg_combined_score
str,i64,f64
"""NON_MATCH""",19028592,0.692759
"""TRUE_MATCH""",1816235,0.868409


In [18]:
combined_stats = con.execute(f"""
SELECT
    label,
    COUNT(*) AS pairs,

    ROUND(AVG(0.35 * name_jw + 0.65 * address_jw), 6) AS avg_score,

    ROUND(
        MEDIAN(0.35 * name_jw + 0.65 * address_jw),
        6
    ) AS median_score

FROM read_parquet('{LABELED_FUZZY}')
GROUP BY label
ORDER BY label
""").pl()

combined_stats

label,pairs,avg_score,median_score
str,i64,f64,f64
"""NON_MATCH""",19028592,0.692759,0.707273
"""TRUE_MATCH""",1816235,0.868409,0.911758


In [19]:
thresholds = [
    0.70, 0.75, 0.80, 0.82,
    0.84, 0.86, 0.88, 0.90,
    0.92, 0.94, 0.96
]

threshold_sql = ", ".join(str(x) for x in thresholds)

threshold_results = con.execute(f"""
WITH scored AS (
    SELECT
        label,
        0.35 * name_jw + 0.65 * address_jw AS score
    FROM read_parquet('{LABELED_FUZZY}')
),
thresholds AS (
    SELECT * FROM (VALUES
        {", ".join(f"({x})" for x in thresholds)}
    ) AS t(threshold)
)
SELECT
    t.threshold,

    SUM(CASE
        WHEN s.label = 'TRUE_MATCH'
         AND s.score >= t.threshold
        THEN 1 ELSE 0
    END) AS true_positives,

    SUM(CASE
        WHEN s.label = 'NON_MATCH'
         AND s.score >= t.threshold
        THEN 1 ELSE 0
    END) AS false_positives,

    SUM(CASE
        WHEN s.label = 'TRUE_MATCH'
         AND s.score < t.threshold
        THEN 1 ELSE 0
    END) AS false_negatives

FROM thresholds t
CROSS JOIN scored s
GROUP BY t.threshold
ORDER BY t.threshold
""").pl()

threshold_results

threshold,true_positives,false_positives,false_negatives
"decimal[3,2]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
0.70,1694140,10775677,122095
0.75,1638041,2725120,178194
0.80,1499748,267805,316487
0.82,1419664,180052,396571
0.84,1313809,149076,502426
…,…,…,…
0.88,1056110,105943,760125
0.90,968595,84920,847640
0.92,852362,63024,963873


In [20]:
threshold_metrics = (
    threshold_results
    .with_columns([
        (
            pl.col("true_positives") /
            (pl.col("true_positives") + pl.col("false_positives"))
        ).alias("precision"),

        (
            pl.col("true_positives") / 3_693_619
        ).alias("recall"),
    ])
    .with_columns(
        (
            1.25 * pl.col("precision") * pl.col("recall") /
            (
                0.25 * pl.col("precision") +
                pl.col("recall")
            )
        ).alias("f0_5")
    )
    .with_columns([
        pl.col("precision").round(4),
        pl.col("recall").round(4),
        pl.col("f0_5").round(4),
    ])
)

threshold_metrics

threshold,true_positives,false_positives,false_negatives,precision,recall,f0_5
"decimal[3,2]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",f64
0.70,1694140,10775677,122095,0,0,NaN
0.75,1638041,2725120,178194,0,0,NaN
0.80,1499748,267805,316487,1,0,0.0
0.82,1419664,180052,396571,1,0,0.0
0.84,1313809,149076,502426,1,0,0.0
…,…,…,…,…,…,…
0.88,1056110,105943,760125,1,0,0.0
0.90,968595,84920,847640,1,0,0.0
0.92,852362,63024,963873,1,0,0.0


In [21]:
threshold_metrics.sort("f0_5", descending=True)

threshold,true_positives,false_positives,false_negatives,precision,recall,f0_5
"decimal[3,2]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",f64
0.70,1694140,10775677,122095,0,0,NaN
0.75,1638041,2725120,178194,0,0,NaN
0.80,1499748,267805,316487,1,0,0.0
0.82,1419664,180052,396571,1,0,0.0
0.84,1313809,149076,502426,1,0,0.0
…,…,…,…,…,…,…
0.88,1056110,105943,760125,1,0,0.0
0.90,968595,84920,847640,1,0,0.0
0.92,852362,63024,963873,1,0,0.0


In [22]:
id="m7x2q9"
threshold_metrics = (
    threshold_results
    .with_columns([
        (
            pl.col("true_positives").cast(pl.Float64) /
            (
                pl.col("true_positives").cast(pl.Float64) +
                pl.col("false_positives").cast(pl.Float64)
            )
        ).alias("precision"),

        (
            pl.col("true_positives").cast(pl.Float64) /
            3_693_619.0
        ).alias("recall"),
    ])
    .with_columns(
        (
            1.25 * pl.col("precision") * pl.col("recall") /
            (
                0.25 * pl.col("precision") +
                pl.col("recall")
            )
        ).alias("f0_5")
    )
    .select([
        "threshold",
        "true_positives",
        "false_positives",
        "false_negatives",
        "precision",
        "recall",
        "f0_5"
    ])
    .with_columns([
        pl.col("precision").round(4),
        pl.col("recall").round(4),
        pl.col("f0_5").round(4),
    ])
)

threshold_metrics

threshold,true_positives,false_positives,false_negatives,precision,recall,f0_5
"decimal[3,2]","decimal[38,0]","decimal[38,0]","decimal[38,0]",f64,f64,f64
0.70,1694140,10775677,122095,0.1359,0.4587,0.1581
0.75,1638041,2725120,178194,0.3754,0.4435,0.3873
0.80,1499748,267805,316487,0.8485,0.406,0.6967
0.82,1419664,180052,396571,0.8874,0.3844,0.7033
0.84,1313809,149076,502426,0.8981,0.3557,0.6882
…,…,…,…,…,…,…
0.88,1056110,105943,760125,0.9088,0.2859,0.633
0.90,968595,84920,847640,0.9194,0.2622,0.6124
0.92,852362,63024,963873,0.9312,0.2308,0.5794


In [23]:
threshold_metrics.sort("f0_5", descending=True)

threshold,true_positives,false_positives,false_negatives,precision,recall,f0_5
"decimal[3,2]","decimal[38,0]","decimal[38,0]","decimal[38,0]",f64,f64,f64
0.82,1419664,180052,396571,0.8874,0.3844,0.7033
0.80,1499748,267805,316487,0.8485,0.406,0.6967
0.84,1313809,149076,502426,0.8981,0.3557,0.6882
0.86,1178154,126947,638081,0.9027,0.319,0.6608
0.88,1056110,105943,760125,0.9088,0.2859,0.633
…,…,…,…,…,…,…
0.92,852362,63024,963873,0.9312,0.2308,0.5794
0.94,676264,42553,1139971,0.9408,0.1831,0.5147
0.96,470446,23767,1345789,0.9519,0.1274,0.4148


In [25]:
COMBINED_V2 = (
    CANDIDATE_DIR /
    "combined_name_translit_candidates.parquet"
)

print("Exists:", COMBINED_V2.exists())
print("Path:", COMBINED_V2)

Exists: True
Path: D:\business_entity_resolution\dataset\work\candidates\combined_name_translit_candidates.parquet


In [26]:
MISSED_PAIRS = CANDIDATE_DIR / "missed_true_pairs.parquet"

con.execute(f"""
COPY (
    SELECT
        g.source1_entity_id,
        g.candidate_entity_id,
        s1.name_norm AS s1_name,
        s2.name_norm AS s2_name,
        s1.address_norm AS s1_address,
        s2.address_norm AS s2_address,
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ) AS name_jw,
        jaro_winkler_similarity(
            s1.address_norm,
            s2.address_norm
        ) AS address_jw
    FROM gt_s2 g
    JOIN train_s1 s1
        ON g.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
        ON g.candidate_entity_id = s2.entity_id
    LEFT JOIN read_parquet('{COMBINED_V2}') c
        ON g.source1_entity_id = c.source1_entity_id
       AND g.candidate_entity_id = c.candidate_entity_id
    WHERE c.candidate_entity_id IS NULL
)
TO '{MISSED_PAIRS}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(MISSED_PAIRS)

D:\business_entity_resolution\dataset\work\candidates\missed_true_pairs.parquet


In [27]:
con.execute(f"""
SELECT *
FROM read_parquet('{MISSED_PAIRS}')
ORDER BY name_jw DESC
LIMIT 30
""").pl()

source1_entity_id,candidate_entity_id,s1_name,s2_name,s1_address,s2_address,name_jw,address_jw
str,str,str,str,str,str,f64,f64
"""S1-977315012""","""S2-931458558""","""urology care inc""","""urology care inc""","""627 clarence avenue bronx ny""","""clarence ave bronx ny""",1.0,0.757937
"""S1-349029896""","""S2-714511179""","""pediatric dental health""","""pediatric dental health""","""13285 rose boulevard northport…","""13285 rose blvd northport al""",1.0,0.919697
"""S1-236331488""","""S2-842225157""","""compass""","""compass""","""26 colby avenue claymont de""","""26 colby ave claymont de""",1.0,0.927778
"""S1-667569750""","""S2-729782241""","""physical therapy specialists""","""physical therapy specialists""","""655 gahle drive westminster md""","""655 1 2 gahle dr westminster m…",1.0,0.887527
"""S1-633207361""","""S2-858466899""","""polaris""","""polaris""","""ny brookhaven 116 exmore court""","""0116 exmore ct riidge ny""",1.0,0.621078
…,…,…,…,…,…,…,…
"""S1-475092230""","""S2-82003140""","""physical therapy health""","""physical therapy health""","""az 61057 flint drive oracle""","""1057 flint dr oracle cdp az""",1.0,0.693122
"""S1-558096594""","""S2-854171877""","""titan""","""titan""","""809 east avenue unit unit 5 ro…","""809 eadt avenue rochester ny""",1.0,0.853783
"""S1-431671491""","""S2-973079978""","""zephyr inc""","""zephyr inc""","""798 washington street charles …","""798 washington street charles …",1.0,1.0


In [28]:
exact_name_missed = con.execute(f"""
SELECT
    COUNT(*) AS missed_pairs,
    SUM(
        CASE
            WHEN s1.name_norm = s2.name_norm
            THEN 1
            ELSE 0
        END
    ) AS exact_name_matches
FROM read_parquet('{MISSED_PAIRS}') m
JOIN train_s1 s1
    ON m.source1_entity_id = s1.entity_id
JOIN train_s2 s2
    ON m.candidate_entity_id = s2.entity_id
""").pl()

exact_name_missed

missed_pairs,exact_name_matches
i64,"decimal[38,0]"
1877384,28801


In [29]:
missed_name_buckets = con.execute(f"""
SELECT
    CASE
        WHEN name_jw >= 0.95 THEN '>= 0.95'
        WHEN name_jw >= 0.90 THEN '0.90 - 0.95'
        WHEN name_jw >= 0.80 THEN '0.80 - 0.90'
        WHEN name_jw >= 0.70 THEN '0.70 - 0.80'
        WHEN name_jw >= 0.50 THEN '0.50 - 0.70'
        ELSE '< 0.50'
    END AS name_similarity,

    COUNT(*) AS missed_pairs

FROM read_parquet('{MISSED_PAIRS}')

GROUP BY 1
ORDER BY
    CASE name_similarity
        WHEN '>= 0.95' THEN 1
        WHEN '0.90 - 0.95' THEN 2
        WHEN '0.80 - 0.90' THEN 3
        WHEN '0.70 - 0.80' THEN 4
        WHEN '0.50 - 0.70' THEN 5
        ELSE 6
    END
""").pl()

missed_name_buckets

name_similarity,missed_pairs
str,i64
""">= 0.95""",395738
"""0.90 - 0.95""",501987
"""0.80 - 0.90""",383089
"""0.70 - 0.80""",123214
"""0.50 - 0.70""",102882
"""< 0.50""",370474


In [30]:
con.execute("""
CREATE OR REPLACE VIEW s1_fuzzy_blocks AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    LENGTH(name_norm) AS name_len,
    LEFT(name_norm, 1) AS name_first
FROM train_s1
WHERE name_norm <> ''
""")

con.execute("""
CREATE OR REPLACE VIEW s2_fuzzy_blocks AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    LENGTH(name_norm) AS name_len,
    LEFT(name_norm, 1) AS name_first
FROM train_s2
WHERE name_norm <> ''
""")

fuzzy_block_stats = con.execute("""
SELECT
    name_first,
    country_norm,
    name_len,
    COUNT(*) AS s2_records
FROM s2_fuzzy_blocks
GROUP BY 1, 2, 3
HAVING COUNT(*) <= 500
ORDER BY s2_records DESC
LIMIT 20
""").pl()

fuzzy_block_stats

name_first,country_norm,name_len,s2_records
str,str,i64,i64
"""ड""","""india""",33,499
"""र""","""india""",23,498
"""a""","""us""",5,497
"""फ""","""india""",33,496
"""b""","""india""",8,496
…,…,…,…
"""इ""","""india""",28,486
"""f""","""india""",39,485
"""अ""","""india""",26,485


In [31]:
con.execute("""
CREATE OR REPLACE VIEW s1_fuzzy_blocks2 AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    LENGTH(name_norm) AS name_len,
    LEFT(name_norm, 1) AS name_first,
    RIGHT(name_norm, 1) AS name_last
FROM train_s1
WHERE name_norm <> ''
""")

con.execute("""
CREATE OR REPLACE VIEW s2_fuzzy_blocks2 AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    LENGTH(name_norm) AS name_len,
    LEFT(name_norm, 1) AS name_first,
    RIGHT(name_norm, 1) AS name_last
FROM train_s2
WHERE name_norm <> ''
""")

block_stats = con.execute("""
SELECT
    name_first,
    name_last,
    country_norm,
    name_len,
    COUNT(*) AS s2_records
FROM s2_fuzzy_blocks2
GROUP BY 1, 2, 3, 4
ORDER BY s2_records DESC
LIMIT 20
""").pl()

block_stats

name_first,name_last,country_norm,name_len,s2_records
str,str,str,i64,i64
"""s""","""d""","""india""",31,6713
"""s""","""d""","""india""",30,6709
"""s""","""d""","""india""",32,6513
"""s""","""d""","""india""",29,6456
"""s""","""d""","""india""",28,6260
…,…,…,…,…
"""p""","""d""","""india""",32,4930
"""p""","""d""","""india""",26,4920
"""s""","""d""","""india""",25,4841


In [32]:
con.execute("""
CREATE OR REPLACE VIEW s1_fuzzy_blocks3 AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    LEFT(name_norm, 3) AS name_prefix3,
    RIGHT(name_norm, 3) AS name_suffix3
FROM train_s1
WHERE LENGTH(name_norm) >= 3
""")

con.execute("""
CREATE OR REPLACE VIEW s2_fuzzy_blocks3 AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    LEFT(name_norm, 3) AS name_prefix3,
    RIGHT(name_norm, 3) AS name_suffix3
FROM train_s2
WHERE LENGTH(name_norm) >= 3
""")

block_stats3 = con.execute("""
SELECT
    name_prefix3,
    name_suffix3,
    country_norm,
    COUNT(*) AS s2_records
FROM s2_fuzzy_blocks3
GROUP BY 1, 2, 3
ORDER BY s2_records DESC
LIMIT 20
""").pl()

block_stats3

name_prefix3,name_suffix3,country_norm,s2_records
str,str,str,i64
"""pri""","""ted""","""india""",28724
"""pvt""","""ltd""","""india""",11919
"""pri""","""ltd""","""india""",11723
"""shr""","""ted""","""india""",9871
"""the""","""llc""","""us""",9679
…,…,…,…
"""ped""","""llc""","""us""",4689
"""ind""","""ltd""","""india""",4514
"""new""","""ted""","""india""",4420


In [33]:
con.execute("""
CREATE OR REPLACE VIEW s1_informative AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND LENGTH(x) >= 3
                    AND x NOT IN (
                        'private','pvt','limited','ltd','llc','llp',
                        'inc','incorporated','corp','corporation',
                        'company','co','lp','pllc'
                    )
            )
        ),
        ' '
    ) AS informative_name
FROM train_s1
WHERE name_norm <> ''
""")

con.execute("""
CREATE OR REPLACE VIEW s2_informative AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND LENGTH(x) >= 3
                    AND x NOT IN (
                        'private','pvt','limited','ltd','llc','llp',
                        'inc','incorporated','corp','corporation',
                        'company','co','lp','pllc'
                    )
            )
        ),
        ' '
    ) AS informative_name
FROM train_s2
WHERE name_norm <> ''
""")

In [34]:
con.execute("""
SELECT
    name_norm,
    informative_name,
    country_norm
FROM s2_informative
WHERE informative_name <> ''
LIMIT 20
""").pl()

name_norm,informative_name,country_norm
str,str,str
"""राम मार्केटिंग प्राइवेट लिमिटे…","""प्राइवेट मार्केटिंग राम लिमिटे…","""india"""
"""holloway peak inc seafood""","""holloway peak seafood""","""us"""
"""आदित्य प्रॉपर्टीज एलएलपी""","""आदित्य एलएलपी प्रॉपर्टीज""","""india"""
"""summit inc""","""summit""","""us"""
"""delta tetlecommunication inc""","""delta tetlecommunication""","""us"""
…,…,…
"""candy s services""","""candy services""","""us"""
"""रियल मॉडर्न फूड लिमिटेड""","""फूड मॉडर्न रियल लिमिटेड""","""india"""
"""crestline crestline clean lp""","""clean crestline crestline""","""us"""


In [36]:
LEGAL_TOKEN_SQL = """
'private','pvt','limited','ltd','llc','llp','inc','incorporated',
'corp','corporation','company','co','lp','pllc',
'प्राइवेट','लिमिटेड','प्रा','लि',
'ప్రైవేట్','లిమిటెడ్',
'ಪ್ರೈವೇಟ್','ಲಿಮಿಟೆಡ್',
'பிரைவேட்','லிமிடெட்',
'প্রাইভেট','লিমিটেড',
'પ્રાઇવેટ','લિમિટેડ'
"""

In [37]:
con.execute(f"""
CREATE OR REPLACE VIEW s1_informative AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND LENGTH(x) >= 3
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
            )
        ),
        ' '
    ) AS informative_name
FROM train_s1
WHERE name_norm <> ''
""")

con.execute(f"""
CREATE OR REPLACE VIEW s2_informative AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND LENGTH(x) >= 3
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
            )
        ),
        ' '
    ) AS informative_name
FROM train_s2
WHERE name_norm <> ''
""")

In [38]:
con.execute("""
SELECT
    name_norm,
    informative_name,
    country_norm
FROM s2_informative
WHERE name_norm IN (
    'राम मार्केटिंग प्राइवेट लिमिटेड',
    'holloway peak inc seafood',
    'आदित्य प्रॉपर्टीज एलएलपी',
    'summit inc'
)
""").pl()

name_norm,informative_name,country_norm
str,str,str
"""राम मार्केटिंग प्राइवेट लिमिटे…","""मार्केटिंग राम""","""india"""
"""holloway peak inc seafood""","""holloway peak seafood""","""us"""
"""आदित्य प्रॉपर्टीज एलएलपी""","""आदित्य एलएलपी प्रॉपर्टीज""","""india"""
"""summit inc""","""summit""","""us"""
"""summit inc""","""summit""","""us"""
…,…,…
"""summit inc""","""summit""","""us"""
"""summit inc""","""summit""","""us"""
"""summit inc""","""summit""","""us"""


In [39]:
con.execute(f"""
CREATE OR REPLACE VIEW s1_informative_anchor AS
SELECT
    entity_id,
    name_norm,
    country_norm,

    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
                    AND LENGTH(x) >= 3
            )
        ),
        ' '
    ) AS informative_name
FROM train_s1
WHERE name_norm <> ''
""")

con.execute(f"""
CREATE OR REPLACE VIEW s2_informative_anchor AS
SELECT
    entity_id,
    name_norm,
    country_norm,

    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
                    AND LENGTH(x) >= 3
            )
        ),
        ' '
    ) AS informative_name
FROM train_s2
WHERE name_norm <> ''
""")

In [41]:
con.execute("""
SELECT
    token,
    LENGTH(token) AS char_len,
    hex(token) AS utf8_hex
FROM (
    SELECT UNNEST(string_split(
        'आदित्य प्रॉपर्टीज एलएलपी',
        ' '
    )) AS token
)
""").pl()

token,char_len,utf8_hex
str,i64,str
"""आदित्य""",6,"""E0A486E0A4A6E0A4BFE0A4A4E0A58D…"
"""प्रॉपर्टीज""",10,"""E0A4AAE0A58DE0A4B0E0A589E0A4AA…"
"""एलएलपी""",6,"""E0A48FE0A4B2E0A48FE0A4B2E0A4AA…"


In [42]:
LEGAL_TOKEN_SQL = """
'private','pvt','limited','ltd','llc','llp','inc','incorporated',
'corp','corporation','company','co','lp','pllc',

'प्राइवेट','लिमिटेड','प्रा','लि','एलएलपी',

'ప్రైవేట్','లిమిటెడ్','ఎల్ఎల్పి',
'ಪ್ರೈವೇಟ್','ಲಿಮಿಟೆಡ್','ಎಲ್ಎಲ್ಪಿ',
'பிரைவேட்','லிமிடெட்','எல்எல்பி',
'প্রাইভেট','লিমিটেড','এলএলপি',
'પ્રાઇવેટ','લિમિટેડ','એલએલપી'
"""

In [43]:
con.execute(f"""
CREATE OR REPLACE VIEW s1_informative_anchor AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
                    AND LENGTH(x) >= 3
            )
        ),
        ' '
    ) AS informative_name
FROM train_s1
WHERE name_norm <> ''
""")

con.execute(f"""
CREATE OR REPLACE VIEW s2_informative_anchor AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
                    AND LENGTH(x) >= 3
            )
        ),
        ' '
    ) AS informative_name
FROM train_s2
WHERE name_norm <> ''
""")

In [44]:
con.execute("""
SELECT
    name_norm,
    informative_name
FROM s2_informative_anchor
WHERE name_norm LIKE '%आदित्य%'
LIMIT 10
""").pl()

name_norm,informative_name
str,str
"""आदित्य प्रॉपर्टीज एलएलपी""","""आदित्य प्रॉपर्टीज"""
"""आदित्य आईटी प्राइवेट लिमिटेड""","""आईटी आदित्य"""
"""सिटी आदित्य एनर्जी लिमिटेड""","""आदित्य एनर्जी सिटी"""
"""आदित्य कंसल्टेंसी रेस्टोरेंट प…","""आदित्य कंसल्टेंसी रेस्टोरेंट"""
"""आदित्य हॉस्पिटैलिटी प्राइवेट ल…","""आदित्य हॉस्पिटैलिटी"""
"""आदित्य मीडिया प्राइवेट लिमिटेड""","""आदित्य मीडिया"""
"""आदित्य एनर्जी ट्रेडिंग प्राइवे…","""आदित्य एनर्जी ट्रेडिंग"""
"""आदित्य सॉल्यूशंस""","""आदित्य सॉल्यूशंस"""
"""आदित्य ट्रेडिंग लिमिटेड""","""आदित्य ट्रेडिंग"""


In [45]:
informative_freq = con.execute("""
SELECT
    informative_name,
    country_norm,
    COUNT(*) AS s2_records
FROM s2_informative_anchor
WHERE informative_name <> ''
GROUP BY informative_name, country_norm
ORDER BY s2_records DESC
LIMIT 30
""").pl()

informative_freq

informative_name,country_norm,s2_records
str,str,i64
"""services""","""india""",2874
"""center""","""india""",1980
"""india""","""india""",1816
"""center""","""us""",1668
"""solutions""","""india""",1438
…,…,…
"""internal medicine""","""us""",515
"""center family""","""us""",514
"""behavioral health""","""us""",510


In [46]:
con.execute("""
CREATE OR REPLACE VIEW s2_informative_tokens AS
SELECT
    entity_id,
    country_norm,
    UNNEST(string_split(informative_name, ' ')) AS token
FROM s2_informative_anchor
WHERE informative_name <> ''
""")

token_freq = con.execute("""
SELECT
    token,
    country_norm,
    COUNT(*) AS s2_records
FROM s2_informative_tokens
GROUP BY token, country_norm
ORDER BY s2_records ASC
LIMIT 30
""").pl()

token_freq

token,country_norm,s2_records
str,str,i64
"""chiroprhctic""","""us""",1
"""6585017533""","""india""",1
"""dentitaoy""","""us""",1
"""neteeu""","""india""",1
"""inustciial""","""india""",1
…,…,…
"""cofsfee""","""us""",1
"""rorn""","""india""",1
"""martinezcapitalasa""","""us""",1


In [47]:
con.execute("""
CREATE OR REPLACE VIEW s2_rare_tokens AS
SELECT
    token,
    country_norm,
    COUNT(*) AS token_freq
FROM s2_informative_tokens
GROUP BY token, country_norm
HAVING COUNT(*) <= 50
""")

rare_token_block_stats = con.execute("""
SELECT
    COUNT(*) AS rare_tokens,
    SUM(token_freq) AS total_postings,
    MAX(token_freq) AS max_posting
FROM s2_rare_tokens
""").pl()

rare_token_block_stats

rare_tokens,total_postings,max_posting
i64,"decimal[38,0]",i64
820684,2021163,50


In [48]:
con.execute("""
SELECT
    COUNT(DISTINCT t.entity_id) AS s2_entities_with_rare_token
FROM s2_informative_tokens t
JOIN s2_rare_tokens r
    ON t.token = r.token
   AND t.country_norm = r.country_norm
""").pl()

s2_entities_with_rare_token
i64
1793292


In [50]:
rare_block_sample = con.execute("""
SELECT
    s1.entity_id AS source1_entity_id,
    COUNT(DISTINCT x.entity_id) AS candidate_count
FROM (
    SELECT *
    FROM s1_informative_anchor
    WHERE informative_name <> ''
    LIMIT 10000
) s1
JOIN (
    SELECT
        x.entity_id,
        x.country_norm,
        x.token
    FROM (
        SELECT
            entity_id,
            country_norm,
            UNNEST(string_split(informative_name, ' ')) AS token
        FROM s2_informative_anchor
        WHERE informative_name <> ''
    ) x
    JOIN s2_rare_tokens r
        ON x.token = r.token
       AND x.country_norm = r.country_norm
) x
    ON s1.country_norm = x.country_norm
   AND list_contains(
       string_split(s1.informative_name, ' '),
       x.token
   )
GROUP BY s1.entity_id
ORDER BY candidate_count DESC
""").pl()

rare_block_sample

source1_entity_id,candidate_count
str,i64
"""S1-354150218""",97
"""S1-332338799""",84
"""S1-544803037""",82
"""S1-171207982""",81
"""S1-836916847""",80
…,…
"""S1-948336320""",1
"""S1-279914306""",1
"""S1-137875047""",1


In [51]:
rare_block_sample.select([
    pl.col("candidate_count").mean().alias("mean"),
    pl.col("candidate_count").median().alias("median"),
    pl.col("candidate_count").quantile(0.95).alias("p95"),
    pl.col("candidate_count").max().alias("max"),
])

mean,median,p95,max
f64,f64,f64,i64
22.624484,20.0,48.0,97


In [ ]:
RARE_TOKEN_CANDIDATES = (
    CANDIDATE_DIR /
    "rare_token_candidates.parquet"
)

con.execute(f"""
COPY (
    SELECT DISTINCT
        s1.entity_id AS source1_entity_id,
        x.entity_id AS candidate_entity_id
    FROM s1_informative_anchor s1
    JOIN (
        SELECT
            x.entity_id,
            x.country_norm,
            x.token
        FROM (
            SELECT
                entity_id,
                country_norm,
                UNNEST(string_split(informative_name, ' ')) AS token
            FROM s2_informative_anchor
            WHERE informative_name <> ''
        ) x
        JOIN s2_rare_tokens r
            ON x.token = r.token
           AND x.country_norm = r.country_norm
    ) x
        ON s1.country_norm = x.country_norm
       AND list_contains(
           string_split(s1.informative_name, ' '),
           x.token
       )
    WHERE s1.informative_name <> ''
)
TO '{RARE_TOKEN_CANDIDATES}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("Created:", RARE_TOKEN_CANDIDATES)

In [16]:
address_separation = con.execute(f"""
SELECT
    label,
    COUNT(*) AS pairs,

    ROUND(AVG(name_jw), 6) AS avg_name_jw,
    ROUND(AVG(address_jw), 6) AS avg_address_jw,

    ROUND(MEDIAN(name_jw), 6) AS median_name_jw,
    ROUND(MEDIAN(address_jw), 6) AS median_address_jw,

    ROUND(QUANTILE_CONT(name_jw, 0.10), 6) AS p10_name_jw,
    ROUND(QUANTILE_CONT(address_jw, 0.10), 6) AS p10_address_jw,

    ROUND(QUANTILE_CONT(name_jw, 0.25), 6) AS p25_name_jw,
    ROUND(QUANTILE_CONT(address_jw, 0.25), 6) AS p25_address_jw,

    ROUND(QUANTILE_CONT(name_jw, 0.75), 6) AS p75_name_jw,
    ROUND(QUANTILE_CONT(address_jw, 0.75), 6) AS p75_address_jw

FROM read_parquet('{LABELED_FUZZY}')

GROUP BY label
ORDER BY label
""").pl()

address_separation

RuntimeError: Query interrupted

In [ ]:
def token_overlap_features(df):
    return df.with_columns([
        pl.col("s1_name")
        .str.split(" ")
        .alias("s1_tokens"),

        pl.col("s2_name")
        .str.split(" ")
        .alias("s2_tokens"),
    ]).with_columns([
        pl.struct(["s1_tokens", "s2_tokens"])
        .map_elements(
            lambda x: (
                len(
                    set(x["s1_tokens"]) &
                    set(x["s2_tokens"])
                )
            ),
            return_dtype=pl.Int32,
        )
        .alias("shared_tokens")
    ])

In [ ]:
missed_sample = token_overlap_features(
    missed_details.head(10000)
)

print(
    missed_sample.select([
        "s1_name",
        "s2_name",
        "shared_tokens",
    ]).head(30)
)

shape: (30, 3)
┌─────────────────────────────────┬─────────────────────────────────┬───────────────┐
│ s1_name                         ┆ s2_name                         ┆ shared_tokens │
│ ---                             ┆ ---                             ┆ ---           │
│ str                             ┆ str                             ┆ i32           │
╞═════════════════════════════════╪═════════════════════════════════╪═══════════════╡
│ maure williams colombier inc    ┆ maure wilblims colombier inc    ┆ 3             │
│ maure williams colombier inc    ┆ maure williams colombier        ┆ 3             │
│ raj investments llp             ┆ ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்ப…           ┆ 0             │
│ dahlia power reliable scientif… ┆ dahlia power reliable           ┆ 3             │
│ dahlia power reliable scientif… ┆ dahlia power reliable scientif… ┆ 4             │
│ …                               ┆ …                               ┆ …             │
│ ap hospitality inc         

In [ ]:
PREFIX_LEN = 6
MAX_POSTINGS = 100

s1 = s1.with_columns(
    pl.col("name_norm")
    .str.slice(0, PREFIX_LEN)
    .alias("name_prefix")
)

s2 = s2.with_columns(
    pl.col("name_norm")
    .str.slice(0, PREFIX_LEN)
    .alias("name_prefix")
)

In [ ]:
s2_prefix_frequency = (
    s2
    .filter(
        pl.col("name_prefix").str.len_chars() >= PREFIX_LEN
    )
    .group_by("name_prefix")
    .agg(
        pl.len().alias("posting_count")
    )
)

print(s2_prefix_frequency.sort("posting_count").head(20))
print(
    s2_prefix_frequency
    .select([
        pl.col("posting_count").mean().alias("mean"),
        pl.col("posting_count").median().alias("median"),
        pl.col("posting_count").quantile(0.95).alias("p95"),
        pl.col("posting_count").max().alias("max"),
    ])
)

shape: (20, 2)
┌─────────────┬───────────────┐
│ name_prefix ┆ posting_count │
│ ---         ┆ ---           │
│ str         ┆ u32           │
╞═════════════╪═══════════════╡
│ díva c      ┆ 1             │
│ ncs pi      ┆ 1             │
│ zidpat      ┆ 1             │
│ a 2 gl      ┆ 1             │
│ red id      ┆ 1             │
│ …           ┆ …             │
│ seam f      ┆ 1             │
│ e1va m      ┆ 1             │
│ marc p      ┆ 1             │
│ rnl en      ┆ 1             │
│ gapure      ┆ 1             │
└─────────────┴───────────────┘
shape: (1, 4)
┌───────────┬────────┬──────┬───────┐
│ mean      ┆ median ┆ p95  ┆ max   │
│ ---       ┆ ---    ┆ ---  ┆ ---   │
│ f64       ┆ f64    ┆ f64  ┆ u32   │
╞═══════════╪════════╪══════╪═══════╡
│ 11.203358 ┆ 1.0    ┆ 25.0 ┆ 39386 │
└───────────┴────────┴──────┴───────┘


In [ ]:
selective_prefixes = (
    s2_prefix_frequency
    .filter(
        pl.col("posting_count") <= MAX_POSTINGS
    )
    .select("name_prefix")
)

print(
    f"Total prefixes: {len(s2_prefix_frequency):,}"
)

print(
    f"Selective prefixes: {len(selective_prefixes):,}"
)
s2_selective = (
    s2
    .join(
        selective_prefixes,
        on="name_prefix",
        how="inner",
    )
    .select([
        "entity_id",
        "name_prefix",
    ])
)

print(
    f"S2 records participating in block: "
    f"{len(s2_selective):,}"
)

Total prefixes: 447,827
Selective prefixes: 442,117
S2 records participating in block: 2,000,270


In [ ]:
#37
s1_selective = (
    s1
    .join(
        selective_prefixes,
        on="name_prefix",
        how="inner",
    )
    .select([
        "entity_id",
        "name_prefix",
    ])
)

print(
    f"S1 records participating in block: "
    f"{len(s1_selective):,}"
)
prefix_candidates = (
    s1_selective
    .join(
        s2_selective,
        on="name_prefix",
        how="inner",
        suffix="_s2",
    )
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("entity_id_s2").alias("candidate_entity_id"),
    ])
    .unique()
)

print(
    f"Prefix candidate pairs: "
    f"{len(prefix_candidates):,}"
)

S1 records participating in block: 806,306


: 

In [ ]:
#38
prefix_recovered = (
    gt_s2
    .join(
        prefix_candidates,
        on=[
            "source1_entity_id",
            "candidate_entity_id",
        ],
        how="inner",
    )
)

prefix_recall = (
    len(prefix_recovered) / len(gt_s2)
)

print(f"True pairs:       {len(gt_s2):,}")
print(f"Recovered pairs:  {len(prefix_recovered):,}")
print(f"Recall:           {prefix_recall:.4%}")

In [ ]:
#36
first_token_candidates = (
    s1
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        "first_token",
    ])
    .filter(pl.col("first_token") != "")
    .join(
        s2.select([
            pl.col("entity_id").alias("candidate_entity_id"),
            "first_token",
        ]),
        on="first_token",
        how="inner",
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id",
    ])
    .unique()
)

print(
    f"First-token candidates: "
    f"{len(first_token_candidates):,}"
)
first_token_stats = (
    first_token_candidates
    .group_by("source1_entity_id")
    .agg(
        pl.len().alias("candidate_count")
    )
)

print(
    first_token_stats.select([
        pl.col("candidate_count").mean().alias("mean"),
        pl.col("candidate_count").median().alias("median"),
        pl.col("candidate_count").quantile(0.95).alias("p95"),
        pl.col("candidate_count").max().alias("max"),
    ])
)

In [ ]:
#34 
s2_tokens = (
    s2
    .select([
        pl.col("entity_id"),
        pl.col("name_norm"),
    ])
    .filter(
        pl.col("name_norm").str.len_chars() > 0
    )
    .with_columns(
        pl.col("name_norm")
        .str.split(" ")
        .alias("tokens")
    )
    .explode("tokens")
    .filter(
        pl.col("tokens").str.len_chars() > 1
    )
    .select([
        "entity_id",
        pl.col("tokens").alias("token"),
    ])
    .unique()
)

print(f"Unique entity-token pairs: {len(s2_tokens):,}")

C:\Users\KRISH\tmp\ipykernel_15144\2285123983.py:16: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("tokens")


Unique entity-token pairs: 17,262,104


In [ ]:
token_frequency = (
    s2_tokens
    .group_by("token")
    .agg(
        pl.len().alias("entity_frequency")
    )
    .sort("entity_frequency")
)

print(token_frequency.head(30))
print(
    token_frequency
    .sort("entity_frequency", descending=True)
    .head(30)
)

shape: (30, 2)
┌───────────────────┬──────────────────┐
│ token             ┆ entity_frequency │
│ ---               ┆ ---              │
│ str               ┆ u32              │
╞═══════════════════╪══════════════════╡
│ technologiesoasis ┆ 1                │
│ svnei             ┆ 1                │
│ acaeey            ┆ 1                │
│ asepabe           ┆ 1                │
│ pivalfe           ┆ 1                │
│ …                 ┆ …                │
│ ti1son            ┆ 1                │
│ eddltgidge        ┆ 1                │
│ mftthws           ┆ 1                │
│ eltcerhibal       ┆ 1                │
│ brtboehrs         ┆ 1                │
└───────────────────┴──────────────────┘
shape: (30, 2)
┌─────────────┬──────────────────┐
│ token       ┆ entity_frequency │
│ ---         ┆ ---              │
│ str         ┆ u32              │
╞═════════════╪══════════════════╡
│ limited     ┆ 528795           │
│ llc         ┆ 528303           │
│ private     ┆ 528131      

In [ ]:
thresholds = [1, 2, 5, 10, 25, 50, 100, 250, 500, 1000]

for threshold in thresholds:
    count = token_frequency.filter(
        pl.col("entity_frequency") <= threshold
    ).height

    print(
        f"Frequency <= {threshold:>4}: "
        f"{count:>10,} tokens"
    )
for threshold in thresholds:
    entities = (
        s2_tokens
        .join(
            token_frequency.filter(
                pl.col("entity_frequency") <= threshold
            ),
            on="token",
            how="inner",
        )
        .select("entity_id")
        .n_unique()
    )

    print(
        f"Frequency <= {threshold:>4}: "
        f"{entities:>10,} S2 entities"
    )

Frequency <=    1:    640,328 tokens
Frequency <=    2:    690,115 tokens
Frequency <=    5:    728,995 tokens
Frequency <=   10:    753,295 tokens
Frequency <=   25:    779,177 tokens
Frequency <=   50:    791,437 tokens
Frequency <=  100:    801,540 tokens
Frequency <=  250:    806,446 tokens
Frequency <=  500:    808,234 tokens
Frequency <= 1000:    809,641 tokens
Frequency <=    1:    631,311 S2 entities
Frequency <=    2:    726,272 S2 entities
Frequency <=    5:    860,412 S2 entities
Frequency <=   10:  1,027,185 S2 entities
Frequency <=   25:  1,389,337 S2 entities
Frequency <=   50:  1,740,222 S2 entities
Frequency <=  100:  2,246,534 S2 entities
Frequency <=  250:  2,732,641 S2 entities
Frequency <=  500:  3,110,893 S2 entities
Frequency <= 1000:  3,495,787 S2 entities


In [ ]:
LEGAL_TOKENS = {
    "limited",
    "llc",
    "private",
    "inc",
    "ltd",
    "llp",
    "incorporated",
    "corporation",
    "corp",
    "company",
    "co",
}


In [ ]:
TOKEN_FREQ_MAX = 100

useful_tokens = (
    token_frequency
    .filter(
        pl.col("entity_frequency") <= TOKEN_FREQ_MAX
    )
    .filter(
        ~pl.col("token").is_in(list(LEGAL_TOKENS))
    )
)

print(f"Useful tokens: {len(useful_tokens):,}")

print(
    useful_tokens
    .sort("entity_frequency", descending=True)
    .head(20)
)

Useful tokens: 801,540
shape: (20, 2)
┌────────────┬──────────────────┐
│ token      ┆ entity_frequency │
│ ---        ┆ ---              │
│ str        ┆ u32              │
╞════════════╪══════════════════╡
│ csr        ┆ 100              │
│ dio        ┆ 100              │
│ reia       ┆ 100              │
│ aguayo     ┆ 100              │
│ rosen      ┆ 100              │
│ …          ┆ …                │
│ ted        ┆ 100              │
│ bhagirathi ┆ 100              │
│ candace    ┆ 100              │
│ ínteriors  ┆ 100              │
│ satnam     ┆ 100              │
└────────────┴──────────────────┘


In [ ]:
s2_rare_token_index = (
    s2_tokens
    .join(
        useful_tokens.select(["token"]),
        on="token",
        how="inner",
    )
)

print(
    f"S2 rare-token postings: "
    f"{len(s2_rare_token_index):,}"
)

S2 rare-token postings: 2,649,803


In [ ]:
s1_tokens = (
    s1
    .select([
        pl.col("entity_id"),
        pl.col("name_norm"),
    ])
    .filter(
        pl.col("name_norm").str.len_chars() > 0
    )
    .with_columns(
        pl.col("name_norm")
        .str.split(" ")
        .alias("tokens")
    )
    .explode("tokens")
    .filter(
        pl.col("tokens").str.len_chars() > 1
    )
    .select([
        "entity_id",
        pl.col("tokens").alias("token"),
    ])
    .unique()
)
s1_rare_tokens = (
    s1_tokens
    .join(
        useful_tokens.select(["token"]),
        on="token",
        how="inner",
    )
)

print(
    f"S1 rare-token postings: "
    f"{len(s1_rare_tokens):,}"
)

C:\Users\KRISH\tmp\ipykernel_15144\3093373060.py:15: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("tokens")


S1 rare-token postings: 814,781


: 

In [ ]:
rare_token_candidates = (
    s1_rare_tokens
    .join(
        s2_rare_token_index,
        on="token",
        how="inner",
        suffix="_s2",
    )
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("entity_id_s2").alias("candidate_entity_id"),
    ])
    .unique()
)

print(
    f"Rare-token candidate pairs: "
    f"{len(rare_token_candidates):,}"
)

In [ ]:
rare_token_recovered = (
    gt_s2
    .join(
        rare_token_candidates,
        on=[
            "source1_entity_id",
            "candidate_entity_id",
        ],
        how="inner",
    )
)

rare_token_recall = (
    len(rare_token_recovered) / len(gt_s2)
)

print(f"True S1→S2 pairs:      {len(gt_s2):,}")
print(f"Recovered pairs:       {len(rare_token_recovered):,}")
print(f"Blocking recall:       {rare_token_recall:.4%}")

In [ ]:
rare_token_stats = (
    rare_token_candidates
    .group_by("source1_entity_id")
    .agg(
        pl.len().alias("candidate_count")
    )
)

print(
    rare_token_stats.select([
        pl.col("candidate_count").mean().alias("mean"),
        pl.col("candidate_count").median().alias("median"),
        pl.col("candidate_count").quantile(0.95).alias("p95"),
        pl.col("candidate_count").max().alias("max"),
    ])
)